In [1]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import MarkdownNodeParser

# 1. LOAD: Read the Markdown files you downloaded from LlamaIndex Cloud
# This looks into your 'data' folder and pulls in all .md files.
reader = SimpleDirectoryReader(input_dir="./data/markdown")
documents = reader.load_data()

# 2. DIVIDE: Use the Markdown headers to split the text logically
# Instead of cutting by character count, it cuts when it sees a new ## or ###
parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(documents)

# 3. VERIFY: Let's see what the "Sections" actually look like
print(f"--- Division Complete ---")
print(f"Total logical sections found: {len(nodes)}\n")

# Peek at the first 3 nodes to see the 'Context' (the headers)
for i, node in enumerate(nodes[:3]):
    # Get the header title from metadata
    header = node.metadata.get("header_path", "Main Title")
    print(f"SECTION {i}: {header}")
    print(f"TEXT PREVIEW: {node.text[:100]}...")
    print("-" * 30)

--- Division Complete ---
Total logical sections found: 165

SECTION 0: /
TEXT PREVIEW: An illustration shows a person lying in bed, covering their face with their hands, with a scribbled ...
------------------------------
SECTION 1: /
TEXT PREVIEW: ## Is it stress or anxiety?

<table>
  <tbody>
    <tr>
        <td>Stress</td>
        <td>Both Str...
------------------------------
SECTION 2: /
TEXT PREVIEW: ## Ways to Cope
* Keep a journal.
* Download an app with relaxation exercises.
* Exercise and eat he...
------------------------------


In [3]:
# ============================================================
# NEO4J CONNECTION SETUP
# ============================================================

from neo4j import GraphDatabase

# Neo4j Aura Credentials
NEO4J_URI      = "neo4j+s://90eb030b.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "16uM7KoQsjawGMGePsslo1EY4omUWMrBGm8LBipugfg"
NEO4J_DATABASE = "neo4j"

# Create driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Test connection
try:
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("RETURN 'connected' AS status")
        print(f"✅ Neo4j connection: {result.single()['status']}")
except Exception as e:
    print(f"❌ Neo4j connection failed: {e}")

print(f"   URI: {NEO4J_URI}")



❌ Neo4j connection failed: {neo4j_code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.} {gql_status: 42NFF} {gql_status_description: error: syntax error or access rule violation - permission/access denied. Access denied, see the security logs for details.}
   URI: neo4j+s://90eb030b.databases.neo4j.io


In [ ]:
# ============================================================
# NEO4J GRAPH WRITE FUNCTIONS
# ============================================================

def write_graph_to_neo4j(driver, custom_entities, seen_relations,
                         relation_metadata, entity_sources, chunk_citation_map=None):
    """
    Writes the full extracted graph to Neo4j.
    Uses MERGE so re-running is safe — no duplicates created.
    """
    if chunk_citation_map is None:
        chunk_citation_map = {}
    
    def _write(tx, entities, relations, rel_meta, ent_sources, citation_map):
        # ── Nodes ─────────────────────────────────────────
        for canon, entity in entities.items():
            sources = list(ent_sources.get(canon, set()))
            citations = [citation_map.get(s, "") for s in sources]
            
            # No-APOC version - stores type as property
            tx.run(
                """
                MERGE (n:Entity {name: $name})
                SET n.type        = $type,
                    n.description = $description,
                    n.sources     = $sources,
                    n.citations   = $citations
                """,
                name=canon,
                type=entity.label,
                description=entity.properties.get("description", ""),
                sources=sources,
                citations=citations,
            )
        
        # ── Edges ─────────────────────────────────────────
        for rel_key in relations:
            if isinstance(rel_key, tuple) and len(rel_key) == 3:
                subj, rel_label, obj = rel_key
            else:
                continue
                
            meta = rel_meta.get((subj, rel_label, obj), {})
            strength = meta.get("strength", 5)
            desc = meta.get("description", "")
            src_list = list(meta.get("sources", set()))
            
            # Dynamic relationship type
            query = f"""
                MATCH (a:Entity {{name: $subj}})
                MATCH (b:Entity {{name: $obj}})
                MERGE (a)-[r:{rel_label}]->(b)
                SET r.strength    = $strength,
                    r.description = $desc,
                    r.sources     = $src_list
            """
            tx.run(query, subj=subj, obj=obj, strength=strength, 
                   desc=desc, src_list=src_list)
    
    with driver.session(database=NEO4J_DATABASE) as session:
        session.execute_write(
            _write,
            custom_entities,
            seen_relations,
            relation_metadata,
            entity_sources,
            chunk_citation_map,
        )
    
    print(f"✅ Written to Neo4j: {len(custom_entities)} nodes, {len(seen_relations)} edges")


def clear_neo4j_graph(driver):
    """Clear all nodes and relationships from Neo4j (use with caution!)"""
    with driver.session(database=NEO4J_DATABASE) as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("✅ Neo4j graph cleared")


def create_neo4j_indexes(driver):
    """Create indexes for fast lookups. Run once after populating graph."""
    with driver.session(database=NEO4J_DATABASE) as session:
        # Unique constraint on entity name
        try:
            session.run(
                "CREATE CONSTRAINT entity_name IF NOT EXISTS "
                "FOR (n:Entity) REQUIRE n.name IS UNIQUE"
            )
            print("✅ Created unique constraint on Entity.name")
        except Exception as e:
            print(f"   Constraint may already exist: {e}")
        
        # Full-text search index
        try:
            session.run(
                """
                CREATE FULLTEXT INDEX entity_search IF NOT EXISTS
                FOR (n:Entity) ON EACH [n.name, n.description]
                """
            )
            print("✅ Created full-text search index")
        except Exception as e:
            print(f"   Full-text index may already exist: {e}")
        
        # Index on type for filtering
        try:
            session.run(
                "CREATE INDEX entity_type IF NOT EXISTS FOR (n:Entity) ON (n.type)"
            )
            print("✅ Created index on Entity.type")
        except Exception as e:
            print(f"   Type index may already exist: {e}")

print("✅ Neo4j write functions loaded")



In [ ]:
# ============================================================
# NEO4J QUERY HELPERS
# Drop-in replacements for local graph queries
# ============================================================

def neo4j_forward_lookup(driver, condition_name: str) -> dict:
    """
    What are the symptoms and treatments of X?
    Returns dict grouped by relation type.
    """
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(
            """
            MATCH (c:Entity {name: $name})-[r]->(target:Entity)
            RETURN type(r) AS relation,
                   target.name AS target,
                   target.type AS target_type,
                   r.strength AS strength,
                   r.description AS description
            ORDER BY r.strength DESC
            """,
            name=condition_name.upper()
        )
        rows = result.data()
    
    grouped = {}
    for row in rows:
        rel = row["relation"]
        grouped.setdefault(rel, []).append({
            "name": row["target"],
            "type": row["target_type"],
            "strength": row["strength"],
            "description": row["description"]
        })
    return grouped


def neo4j_reverse_lookup(driver, symptom_names: list) -> list:
    """
    Which conditions have these symptoms? (differential diagnosis)
    Returns list of conditions ranked by match count.
    """
    symptom_names = [s.upper() for s in symptom_names]
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(
            """
            MATCH (c:Entity)-[:HAS_SYMPTOM]->(s:Entity)
            WHERE s.name IN $symptoms
              AND c.type = 'CONDITION'
            RETURN c.name AS condition,
                   c.description AS description,
                   collect(s.name) AS matched_symptoms,
                   count(s) AS match_count
            ORDER BY match_count DESC
            """,
            symptoms=symptom_names
        )
        return result.data()


def neo4j_comparison(driver, condition_a: str, condition_b: str) -> dict:
    """
    What do two conditions share vs differ on?
    Returns dict with 'shared' and 'unique_a' and 'unique_b' keys.
    """
    condition_a = condition_a.upper()
    condition_b = condition_b.upper()
    
    result_dict = {"shared": {}, "unique_a": {}, "unique_b": {}}
    
    with driver.session(database=NEO4J_DATABASE) as session:
        # Find shared connections
        shared_result = session.run(
            """
            MATCH (ca:Entity {name: $a})-[ra]->(shared:Entity)<-[rb]-(cb:Entity {name: $b})
            WHERE type(ra) = type(rb)
            RETURN type(ra) AS relation, shared.name AS entity, shared.type AS type
            """,
            a=condition_a, b=condition_b
        )
        for row in shared_result.data():
            rel = row["relation"]
            result_dict["shared"].setdefault(rel, set()).add(row["entity"])
        
        # Find unique to A
        unique_a_result = session.run(
            """
            MATCH (ca:Entity {name: $a})-[r]->(unique_a:Entity)
            WHERE NOT EXISTS {
                MATCH (cb:Entity {name: $b})-[]->(unique_a)
            }
            RETURN type(r) AS relation, unique_a.name AS entity, unique_a.type AS type
            """,
            a=condition_a, b=condition_b
        )
        for row in unique_a_result.data():
            rel = row["relation"]
            result_dict["unique_a"].setdefault(rel, set()).add(row["entity"])
        
        # Find unique to B
        unique_b_result = session.run(
            """
            MATCH (cb:Entity {name: $b})-[r]->(unique_b:Entity)
            WHERE NOT EXISTS {
                MATCH (ca:Entity {name: $a})-[]->(unique_b)
            }
            RETURN type(r) AS relation, unique_b.name AS entity, unique_b.type AS type
            """,
            a=condition_a, b=condition_b
        )
        for row in unique_b_result.data():
            rel = row["relation"]
            result_dict["unique_b"].setdefault(rel, set()).add(row["entity"])
    
    return result_dict


def neo4j_text_query(driver, keywords: list, limit: int = 20) -> list:
    """
    Full-text keyword search across all node names and descriptions.
    Falls back to CONTAINS if full-text index doesn't exist.
    """
    try:
        query_string = " OR ".join(keywords)
        with driver.session(database=NEO4J_DATABASE) as session:
            result = session.run(
                """
                CALL db.index.fulltext.queryNodes('entity_search', $q)
                YIELD node, score
                RETURN node.name AS name, node.type AS type,
                       node.description AS description, score
                ORDER BY score DESC
                LIMIT $limit
                """,
                q=query_string, limit=limit
            )
            return result.data()
    except Exception:
        # Fallback to simple CONTAINS search
        with driver.session(database=NEO4J_DATABASE) as session:
            keyword_conditions = " OR ".join([f"toLower(n.name) CONTAINS toLower('{kw}')" for kw in keywords])
            result = session.run(
                f"""
                MATCH (n:Entity)
                WHERE {keyword_conditions}
                RETURN n.name AS name, n.type AS type,
                       n.description AS description, 1.0 AS score
                LIMIT $limit
                """,
                limit=limit
            )
            return result.data()


def neo4j_get_all_entities(driver) -> list:
    """Get all entities from Neo4j."""
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(
            """
            MATCH (n:Entity)
            RETURN n.name AS name, n.type AS type, n.description AS description
            ORDER BY n.name
            """
        )
        return result.data()


def neo4j_get_all_relations(driver) -> list:
    """Get all relations from Neo4j."""
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(
            """
            MATCH (a:Entity)-[r]->(b:Entity)
            RETURN a.name AS source, type(r) AS relation, b.name AS target,
                   r.strength AS strength, r.description AS description
            ORDER BY a.name, type(r)
            """
        )
        return result.data()


def neo4j_get_entity_neighbors(driver, entity_name: str) -> dict:
    """Get all neighbors (incoming and outgoing) of an entity."""
    entity_name = entity_name.upper()
    
    with driver.session(database=NEO4J_DATABASE) as session:
        # Outgoing
        out_result = session.run(
            """
            MATCH (n:Entity {name: $name})-[r]->(target:Entity)
            RETURN type(r) AS relation, target.name AS neighbor, 
                   target.type AS type, 'outgoing' AS direction
            """,
            name=entity_name
        )
        outgoing = out_result.data()
        
        # Incoming
        in_result = session.run(
            """
            MATCH (source:Entity)-[r]->(n:Entity {name: $name})
            RETURN type(r) AS relation, source.name AS neighbor,
                   source.type AS type, 'incoming' AS direction
            """,
            name=entity_name
        )
        incoming = in_result.data()
    
    return {"outgoing": outgoing, "incoming": incoming}


print("✅ Neo4j query helpers loaded")



In [ ]:
# ============================================================
# RESTORE FROM NEO4J
# Run this instead of loading pickles - rebuilds in-memory structures
# ============================================================

def restore_from_neo4j(driver):
    """
    Rebuild custom_entities, entity_id_to_node, custom_relations, 
    and relation_metadata from Neo4j.
    """
    from llama_index.core.graph_stores.types import EntityNode, Relation
    
    custom_entities = {}
    entity_id_to_node = {}
    entity_descriptions = {}
    entity_sources = {}
    custom_relations = []
    relation_metadata = {}
    
    with driver.session(database=NEO4J_DATABASE) as session:
        # Load nodes
        nodes = session.run(
            """
            MATCH (n:Entity)
            RETURN n.name AS name, n.type AS type, 
                   n.description AS desc, n.sources AS sources
            """
        ).data()
        
        for row in nodes:
            name = row["name"]
            node = EntityNode(
                name=name,
                label=row["type"],
                properties={
                    "description": row["desc"] or "",
                    "sources": row["sources"] or []
                }
            )
            custom_entities[name] = node
            entity_id_to_node[node.id] = node
            entity_descriptions[name] = row["desc"] or ""
            entity_sources[name] = set(row["sources"] or [])
        
        # Load edges
        edges = session.run(
            """
            MATCH (a:Entity)-[r]->(b:Entity)
            RETURN a.name AS subj, type(r) AS rel, b.name AS obj,
                   r.strength AS strength, r.description AS desc,
                   r.sources AS sources
            """
        ).data()
        
        for row in edges:
            subj, rel, obj = row["subj"], row["rel"], row["obj"]
            
            if subj in custom_entities and obj in custom_entities:
                relation = Relation(
                    source_id=custom_entities[subj].id,
                    target_id=custom_entities[obj].id,
                    label=rel
                )
                custom_relations.append(relation)
                
                relation_metadata[(subj, rel, obj)] = {
                    "description": row["desc"] or "",
                    "strength": row["strength"] or 5,
                    "sources": set(row["sources"] or [])
                }
    
    print(f"✅ Restored from Neo4j:")
    print(f"   Entities:  {len(custom_entities)}")
    print(f"   Relations: {len(custom_relations)}")
    
    return (custom_entities, entity_id_to_node, custom_relations, 
            relation_metadata, entity_descriptions, entity_sources)


# Uncomment to restore from Neo4j instead of pickles:
# (custom_entities, entity_id_to_node, custom_relations, 
#  relation_metadata, entity_descriptions, entity_sources) = restore_from_neo4j(driver)

print("✅ Neo4j restore function loaded")
print("   Call restore_from_neo4j(driver) to load data from Neo4j")



* Now that we have our Nodes (sections like "Symptoms" or "Diagnosis"), we face a classic RAG problem: Context Loss. If the AI finds a node that says "Treatment involves CBT and SSRIs," but it doesn't know that node came from the Social Anxiety brochure, it might accidentally use that answer for PTSD.

* Contextual Enrichment solves this by using an LLM to "stamp" each node with its identity. We are basically "gluing" the metadata directly into the text so that when we build our Knowledge Graph in the next phase, the relationships (Triplets) are 100% accurate.

In [1]:
import os 

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

### Initialize the LLM

In [2]:
from llama_index.llms.openai import OpenAI

# 1. Initialize the "Contextualizer" LLM
# We use a fast, cost-effective model for this (like GPT-4o-mini)
llm = OpenAI(model="gpt-4o-mini", 
             api_key=api_key)

In [7]:
# 2. Define the Enrichment Function
def enrich_nodes_with_context(nodes, llm_client=None, preview_chars=700, progress_every=25):
    import copy
    import re
    from pathlib import Path

    llm_client = llm_client or llm
    progress_every = max(1, int(progress_every))
    enriched_nodes = []
    prompt_cache = {}

    def _clean_doc_name(raw_value):
        if not raw_value:
            return "Unknown Document"
        name = Path(str(raw_value)).name
        name = re.sub(r"\.filtered(?=\.md$)", "", name, flags=re.IGNORECASE)
        name = re.sub(r"\.(md|markdown|txt)$", "", name, flags=re.IGNORECASE)
        return name.replace("_", " ").strip() or "Unknown Document"

    def _extract_section_title(node_text):
        for line in node_text.splitlines():
            stripped = line.strip()
            if not stripped or stripped.startswith("[CONTEXT:"):
                continue
            if stripped.startswith("#"):
                title = re.sub(r"^#+\s*", "", stripped)
            else:
                title = stripped
            title = re.sub(r"[*`_]+", "", title).strip(" -:/")
            if title:
                return title[:120]
        return "General Section"

    def _strip_existing_context(node_text):
        if not node_text.startswith("[CONTEXT:"):
            return node_text
        lines = node_text.splitlines()
        keep_from = 0
        for idx, line in enumerate(lines):
            if idx == 0 and line.startswith("[CONTEXT:"):
                continue
            if keep_from == 0 and line.strip() == "":
                continue
            keep_from = idx
            break
        return "\n".join(lines[keep_from:]).lstrip()

    def _fallback_context(source_doc, section_title):
        if section_title and section_title != "General Section":
            return (
                f"This passage is from {source_doc} and focuses on {section_title} in a mental health reference."
            )
        return f"This passage is from {source_doc} and provides mental health reference context."

    print(f"Starting enrichment for {len(nodes)} nodes...")

    for idx, node in enumerate(nodes, start=1):
        enriched_node = node.model_copy(deep=True) if hasattr(node, "model_copy") else copy.deepcopy(node)

        base_text = enriched_node.metadata.get("original_text", enriched_node.text)
        base_text = _strip_existing_context(base_text)

        source_doc = _clean_doc_name(
            enriched_node.metadata.get("file_name") or enriched_node.metadata.get("file_path")
        )
        header_path = enriched_node.metadata.get("header_path")
        section_title = header_path if header_path and header_path != "/" else _extract_section_title(base_text)
        citation = f"{source_doc} — {section_title}" if section_title and section_title != "General Section" else source_doc

        excerpt = re.sub(r"\s+", " ", base_text).strip()[:preview_chars]
        cache_key = (source_doc, section_title, excerpt)

        if cache_key in prompt_cache:
            context_tag = prompt_cache[cache_key]
        else:
            prompt = f"""You are labeling chunks for mental-health retrieval and graph extraction.
Document: {source_doc}
Section: {section_title}

Write exactly one sentence, under 25 words, that states the disorder/topic and the subtopic of this chunk.
Use the excerpt faithfully. If the disorder is unclear, describe the clinical domain instead of guessing.
Do not mention file names, markdown, or say 'this text', 'this chunk', or 'this passage'.

Excerpt:
{excerpt}

Sentence:"""
            try:
                response = llm_client.complete(prompt)
                context_tag = " ".join((response.text or "").split())
            except Exception as exc:
                context_tag = _fallback_context(source_doc, section_title)
                print(f"  [fallback] Node {idx}: {exc}")

            if not context_tag:
                context_tag = _fallback_context(source_doc, section_title)
            if context_tag[-1:] not in ".!?":
                context_tag += "."
            prompt_cache[cache_key] = context_tag

        enriched_node.metadata["original_text"] = base_text
        enriched_node.metadata["context_tag"] = context_tag
        enriched_node.metadata["section_title"] = section_title
        enriched_node.metadata["source_label"] = source_doc
        enriched_node.metadata["citation"] = citation
        enriched_node.metadata["context_enriched"] = True
        enriched_node.text = f"[CONTEXT: {context_tag}]\n\n{base_text}"

        enriched_nodes.append(enriched_node)
        if idx == 1 or idx % progress_every == 0 or idx == len(nodes):
            print(f"[{idx}/{len(nodes)}] {citation}: {context_tag[:80]}")

    return enriched_nodes

# 3. EXECUTE
enriched_nodes = enrich_nodes_with_context(nodes)

Starting enrichment for 165 nodes...
Enriched: This text belongs to the topic of Symptoms related...
Enriched: This text belongs to the topic of "Symptoms" relat...
Enriched: This text belongs to the sub-topic "Ways to Cope" ...
Enriched: This text belongs to the topic of "Treatment" for ...
Enriched: This text belongs to the topic of Seasonal Affecti...
Enriched: This text belongs to the topic of "Symptoms" relat...
Enriched: This text belongs to the disorder Seasonal Affecti...
Enriched: This text belongs to the disorder Seasonal Affecti...
Enriched: This text belongs to the disorder Seasonal Affecti...
Enriched: This text discusses the causes of Seasonal Affecti...
Enriched: This text belongs to the disorder Seasonal Affecti...
Enriched: This text discusses the treatment of Seasonal Affe...
Enriched: This text discusses the treatment of Seasonal Affe...
Enriched: This text belongs to the disorder of Seasonal Affe...
Enriched: This text discusses the potential role of vitamin ...
Enr

* in the response we want to display the source of the information (source citation)

## Save Cell

In [8]:
import pickle
import os

print("💾 Creating Checkpoints...")
# Create a folder to hold your saved states
os.makedirs("./checkpoints", exist_ok=True)

# 1. Save your Enriched Nodes (So you never have to run enrichment again)
with open("./checkpoints/enriched_nodes.pkl", "wb") as f:
    pickle.dump(enriched_nodes, f)
    print("✅ Enriched Nodes saved to disk!")


💾 Creating Checkpoints...
✅ Enriched Nodes saved to disk!


## Load Cell

In [9]:
import pickle
import os

print("🔄 Loading Checkpoints from disk...")

# 1. Load the Enriched Nodes
if os.path.exists("./checkpoints/enriched_nodes.pkl"):
    with open("./checkpoints/enriched_nodes.pkl", "rb") as f:
        enriched_nodes = pickle.load(f)
    print(f"✅ Loaded {len(enriched_nodes)} enriched nodes!")
else:
    print("⚠️ No enriched nodes checkpoint found.")
    
    
# ============================================================
# BUILD CITATION MAP
# Maps each chunk UUID → "filename — section header"
# so we can show users where each fact came from.
# ============================================================
chunk_citation_map = {}
for node in enriched_nodes:
    chunk_id  = node.node_id
    doc_name  = node.metadata.get("file_name", "Unknown Document")
    # MarkdownNodeParser stores the header path like "## Symptoms > ### Treatment"
    # Try a few common metadata key names it might use
    header = (
        node.metadata.get("header_path") or
        node.metadata.get("section")     or
        node.metadata.get("Header 1")    or
        node.metadata.get("Header 2")    or
        ""
    )
    citation = f"{doc_name} — {header}" if header else doc_name
    chunk_citation_map[chunk_id] = citation

print(f"✅ Built chunk_citation_map for {len(chunk_citation_map)} chunks")

🔄 Loading Checkpoints from disk...
✅ Loaded 165 enriched nodes!
✅ Built chunk_citation_map for 165 chunks


In [15]:
chunk_citation_map

{'2e270990-1477-419c-ac52-f612db9b5901': 'output (1).md — /',
 'db6fea32-ec12-4003-9210-193f5f15d574': 'output (1).md — /',
 '13ad4dd9-ff01-433a-9a4d-21537bcbd37c': 'output (1).md — /',
 '26f8f1ba-d3ea-41ca-9ed1-1f26f67fb62e': 'output (1).md — /',
 '1bc16a44-0f00-4b72-ae0c-a8d1e2f7943c': 'output (10).md — /',
 'e26a2ab0-24dc-4442-b633-c5e6522914ee': 'output (10).md — /Seasonal Affective Disorder/',
 '119507aa-dd81-46af-8338-fdc9b8558756': 'output (10).md — /Seasonal Affective Disorder/',
 'df05df71-9c71-47ad-8edf-5809098a0e2b': 'output (10).md — /Seasonal Affective Disorder/',
 '7205bb9b-6763-4bf9-889a-c064e5f91ec2': 'output (10).md — /Seasonal Affective Disorder/',
 'd77579c1-8552-4430-b305-cefea6314ecf': 'output (10).md — /Seasonal Affective Disorder/',
 '40649d26-fcf4-4eb9-b8f8-4284cef567bb': 'output (10).md — /',
 '1481f317-666f-40da-8485-7e7831dc4848': 'output (10).md — /How is SAD treated?/',
 'c35ae0f9-1eee-47db-a926-aecb6b038783': 'output (10).md — /How is SAD treated?/',
 '2dd

In [ ]:
# ============================================================
# CELL 13B — CHECK GRAPH EXTRACTION PROGRESS (RESUMABLE)
# Checks existing progress and sets start index for resume
# ============================================================

import os
import json

GRAPH_PROGRESS_FILE = "./checkpoints/graph_progress.jsonl"
os.makedirs("./checkpoints", exist_ok=True)

already_processed = 0
pre_loaded_triplets = []

if os.path.exists(GRAPH_PROGRESS_FILE):
    with open(GRAPH_PROGRESS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                already_processed += 1
                pre_loaded_triplets.extend(record.get("triplets", []))
    print(f"Progress file found: {already_processed} / {len(enriched_nodes)} chunks processed")
    print(f"Pre-loaded triplets: {len(pre_loaded_triplets)}")
else:
    print("No graph progress file found. Will start from scratch.")

# RESUME CONTROL - Set to None to auto-resume, or specific index to restart
GRAPH_START_INDEX = None  # <-- change if needed

if GRAPH_START_INDEX is None:
    GRAPH_START_INDEX = already_processed

if GRAPH_START_INDEX == 0 and os.path.exists(GRAPH_PROGRESS_FILE):
    os.remove(GRAPH_PROGRESS_FILE)
    pre_loaded_triplets = []
    print("⚠️ Restart requested — progress file cleared.")

print(f"\n--- Graph extraction will START from chunk: {GRAPH_START_INDEX}")
print(f"--- Chunks remaining: {len(enriched_nodes) - GRAPH_START_INDEX}")

# Graph construction + Triplets extraction

In [38]:
# ============================================================
# CANONICALIZATION UTILITIES — V3
# Root-cause fixes for wrong clinical facts.
# ============================================================

import re

ABBREVIATION_MAP = {
    "CBT":   "COGNITIVE BEHAVIORAL THERAPY",
    "DBT":   "DIALECTICAL BEHAVIOR THERAPY",
    "ACT":   "ACCEPTANCE AND COMMITMENT THERAPY",
    "EMDR":  "EYE MOVEMENT DESENSITIZATION AND REPROCESSING",
    "ECT":   "ELECTROCONVULSIVE THERAPY",
    "TMS":   "TRANSCRANIAL MAGNETIC STIMULATION",
    "SSRI":  "SELECTIVE SEROTONIN REUPTAKE INHIBITOR",
    "SNRI":  "SEROTONIN NOREPINEPHRINE REUPTAKE INHIBITOR",
    "TCA":   "TRICYCLIC ANTIDEPRESSANT",
    "MAOI":  "MONOAMINE OXIDASE INHIBITOR",
    "MDD":   "MAJOR DEPRESSIVE DISORDER",
    "GAD":   "GENERALIZED ANXIETY DISORDER",
    "PTSD":  "POST TRAUMATIC STRESS DISORDER",
    "OCD":   "OBSESSIVE COMPULSIVE DISORDER",
    "BPD":   "BORDERLINE PERSONALITY DISORDER",
    "ADHD":  "ATTENTION DEFICIT HYPERACTIVITY DISORDER",
    "SAD":   "SEASONAL AFFECTIVE DISORDER",
    "BDD":   "BODY DYSMORPHIC DISORDER",
    "ASD":   "AUTISM SPECTRUM DISORDER",
}

# Explicit singularization: only known safe pairs.
# The old regex approach was too aggressive (STRESS → STRES, etc.)
# Maps only the LAST word of a multi-word entity to its singular form.
# Safer than full-string lookup because it handles "MOOD STABILIZERS",
# "ATYPICAL ANTIPSYCHOTICS", "COGNITIVE BEHAVIORAL THERAPIES", etc.
LAST_WORD_SINGULAR = {
    "SYMPTOMS":         "SYMPTOM",
    "CONDITIONS":       "CONDITION",
    "TREATMENTS":       "TREATMENT",
    "MEDICATIONS":      "MEDICATION",
    "THERAPIES":        "THERAPY",
    "DISORDERS":        "DISORDER",
    "EPISODES":         "EPISODE",
    "BEHAVIORS":        "BEHAVIOR",
    "BEHAVIOURS":       "BEHAVIOUR",
    "THOUGHTS":         "THOUGHT",
    "CHANGES":          "CHANGE",
    "PROBLEMS":         "PROBLEM",
    "DIFFICULTIES":     "DIFFICULTY",
    "DISTURBANCES":     "DISTURBANCE",
    "STABILIZERS":      "STABILIZER",
    "INHIBITORS":       "INHIBITOR",
    "ANTIDEPRESSANTS":  "ANTIDEPRESSANT",
    "ANTIPSYCHOTICS":   "ANTIPSYCHOTIC",
    "RESOURCES":        "RESOURCE",
    "HALLUCINATIONS":   "HALLUCINATION",
    "DELUSIONS":        "DELUSION",
    "FLASHBACKS":       "FLASHBACK",
    "NIGHTMARES":       "NIGHTMARE",
    "ATTACKS":          "ATTACK",
    "FEELINGS":         "FEELING",
    "EXPERIENCES":      "EXPERIENCE",
    "DISORDERS":        "DISORDER",
    "ISSUES":           "ISSUE",
    "EVENTS":           "EVENT",
    "TECHNIQUES":       "TECHNIQUE",
    "APPROACHES":       "APPROACH",
}

def canonicalize(name: str) -> str:
    name = name.strip().upper()
    # Normalize separators — hyphens and underscores both become spaces
    name = re.sub(r"[-_]+", " ", name)
    # Collapse multiple spaces
    name = " ".join(name.split())
    # Expand known abbreviations
    name = ABBREVIATION_MAP.get(name, name)
    # Singularize the last word (handles "MOOD STABILIZERS", "ATYPICAL ANTIPSYCHOTICS", etc.)
    words = name.split()
    if words:
        words[-1] = LAST_WORD_SINGULAR.get(words[-1], words[-1])
        name = " ".join(words)
    return name


# ============================================================
# Type priority resolver — unchanged, still correct
# ============================================================
TYPE_PRIORITY = {
    "CONDITION":       6,
    "MEDICATION":      5,
    "TREATMENT":       4,
    "SYMPTOM":         3,
    "DEMOGRAPHIC":     2,
    "SAFETY_RESOURCE": 1,
}

def resolve_type(existing_type: str, new_type: str) -> str:
    return existing_type if TYPE_PRIORITY.get(existing_type, 0) >= TYPE_PRIORITY.get(new_type, 0) else new_type


# ============================================================
# Garbage entity detector — extended
# ============================================================
GARBAGE_FRAGMENTS = {
    "COMBINATION", "VARIOUS", "MULTIPLE", "SEVERAL",
    "LIFELONG", "ONGOING", "CONTINUED", "ADDITIONAL",
    "APPROACH", "OPTION", "STRATEGY", "PROGRAM",
    "INTERVENTION", "MANAGEMENT", "SUPPORT", "SERVICES",
    "CARE", "HELP", "FACTOR", "FACTORS", "ISSUE", "ISSUES",
    "SYMPTOM", "SYMPTOMS", "CONDITION", "CONDITIONS",
    "TREATMENT", "TREATMENTS", "MEDICATION", "MEDICATIONS",
    # Literal placeholder words the LLM emits when it has nothing real
    "INFORMATION", "TOPIC", "ASPECT", "THING", "THINGS",
    "LEVEL", "LEVELS",
}

_STOP_WORDS = {"OF", "AND", "OR", "THE", "A", "AN", "WITH", "FOR", "IN", "TO"}

def is_garbage_entity(name: str) -> bool:
    words = set(name.upper().split()) - _STOP_WORDS
    if not words:
        return True
    if len(name) <= 2:   # single letters / numbers
        return True
    if words.issubset(GARBAGE_FRAGMENTS):
        return True
    if len(words) <= 2 and words & GARBAGE_FRAGMENTS:
        return True
    return False


# ============================================================
# SEMANTIC RELATION RULES — V3
#
# The key upgrade: we now also check the OBJECT's type,
# not just the subject's. This blocks:
#   CONDITION -[HAS_SYMPTOM]-> MEDICATION   (Facts 20-22)
#   CONDITION -[HAS_SYMPTOM]-> DEMOGRAPHIC  (Facts 141-144, 313-314)
#   CONDITION -[HAS_SYMPTOM]-> TREATMENT    (Facts 215, 216, 262, 263)
# ============================================================

# (subject_type, relation, object_type) → allowed
VALID_TRIPLET_TYPES = {
    # A condition has symptoms (which are SYMPTOMS, not medications/demographics)
    ("CONDITION",  "HAS_SYMPTOM",           "SYMPTOM"),
    # A condition can also have another condition as a symptom/comorbidity
    # — we keep this but it's narrow: only CONDITION→CONDITION is allowed,
    # and the extraction prompt now tells the LLM when to use it.
    ("CONDITION",  "HAS_SYMPTOM",           "CONDITION"),

    # Treatments and medications treat conditions
    ("CONDITION",  "TREATED_BY",            "TREATMENT"),
    ("CONDITION",  "TREATED_BY",            "MEDICATION"),

    # A medication is prescribed (source=condition, target=medication) — same as TREATED_BY
    ("CONDITION",  "PRESCRIBES",            "MEDICATION"),

    # Suitable_for: condition/medication/treatment → demographic or condition
    ("CONDITION",  "SUITABLE_FOR",          "DEMOGRAPHIC"),
    ("CONDITION",  "SUITABLE_FOR",          "CONDITION"),
    ("MEDICATION", "SUITABLE_FOR",          "DEMOGRAPHIC"),
    ("MEDICATION", "SUITABLE_FOR",          "CONDITION"),
    ("TREATMENT",  "SUITABLE_FOR",          "DEMOGRAPHIC"),
    ("TREATMENT",  "SUITABLE_FOR",          "CONDITION"),

    # Contraindicated
    ("CONDITION",  "CONTRAINDICATED_FOR",   "DEMOGRAPHIC"),
    ("MEDICATION", "CONTRAINDICATED_FOR",   "DEMOGRAPHIC"),
    ("TREATMENT",  "CONTRAINDICATED_FOR",   "DEMOGRAPHIC"),

    # Crisis / urgent actions
    ("CONDITION",  "URGENT_ACTION",         "SAFETY_RESOURCE"),
    ("SYMPTOM",    "URGENT_ACTION",         "SAFETY_RESOURCE"),
    ("SAFETY_RESOURCE", "URGENT_ACTION",    "SAFETY_RESOURCE"),
}

def is_valid_triplet(subj_type: str, relation: str, obj_type: str) -> bool:
    """
    Check that the (subject_type, relation, object_type) triple is semantically valid.
    This is much stronger than only checking the subject type.
    """
    return (subj_type, relation, obj_type) in VALID_TRIPLET_TYPES

# Keep the old signature for any code that still calls is_valid_relation
def is_valid_relation(subj_type: str, relation: str) -> bool:
    """Legacy shim — prefer is_valid_triplet where object type is known."""
    return any(
        (subj_type, relation, obj) in VALID_TRIPLET_TYPES
        for obj in ("SYMPTOM", "CONDITION", "TREATMENT", "MEDICATION",
                    "DEMOGRAPHIC", "SAFETY_RESOURCE")
    )

# ============================================================
# Smoke tests
# ============================================================
assert canonicalize("MOOD STABILIZERS")   == "MOOD STABILIZER"
assert canonicalize("SYMPTOMS")           == "SYMPTOM"
assert canonicalize("THERAPIES")          == "THERAPY"
assert canonicalize("POST-TRAUMATIC STRESS DISORDER") == "POST TRAUMATIC STRESS DISORDER"
assert canonicalize("POST_TRAUMATIC_STRESS_DISORDER") == "POST TRAUMATIC STRESS DISORDER"
assert is_garbage_entity("COMBINATION OF TREATMENTS") == True
assert is_garbage_entity("LIFELONG TREATMENT")        == True
assert is_garbage_entity("COGNITIVE BEHAVIORAL THERAPY") == False
assert is_garbage_entity("LITHIUM")                   == False
assert is_valid_triplet("CONDITION",  "HAS_SYMPTOM", "SYMPTOM")    == True
assert is_valid_triplet("CONDITION",  "HAS_SYMPTOM", "MEDICATION")  == False  # blocks Facts 20-22
assert is_valid_triplet("CONDITION",  "HAS_SYMPTOM", "DEMOGRAPHIC") == False  # blocks Facts 141-144
assert is_valid_triplet("CONDITION",  "HAS_SYMPTOM", "TREATMENT")   == False  # blocks Facts 215-216
assert is_valid_triplet("CONDITION",  "TREATED_BY",  "TREATMENT")   == True
assert is_valid_triplet("CONDITION",  "TREATED_BY",  "MEDICATION")  == True
assert is_valid_relation("TREATMENT", "HAS_SYMPTOM") == False

print("✅ All canonicalization and validation utilities loaded and tested.")

✅ All canonicalization and validation utilities loaded and tested.


In [39]:
# ============================================================
# CELL 14 — GRAPH EXTRACTION V4 (improved)
# Adds: bidirectional relation index, chunk citation map,
# entity descriptions, source tracking, relation metadata.
# ============================================================

import re
import json
from llama_index.core.llms import ChatMessage
from llama_index.core import PropertyGraphIndex
from llama_index.core.graph_stores.types import EntityNode, Relation

print("--- GRAPH BUILDER V4 (with bidirectional index) ---")

ALLOWED_TYPES = {
    "CONDITION", "SYMPTOM", "TREATMENT",
    "MEDICATION", "SAFETY_RESOURCE", "DEMOGRAPHIC"
}
ALLOWED_RELATIONS = {
    "HAS_SYMPTOM", "TREATED_BY", "PRESCRIBES",
    "SUITABLE_FOR", "CONTRAINDICATED_FOR", "URGENT_ACTION"
}

extraction_prompt = """
You are a medical knowledge graph extractor for mental health literature.
Your job is to extract ONLY the relations that the text explicitly states.
Do NOT infer, generalise, or add clinical knowledge you weren't given.

STEP 1 — Identify the section topic.
The text starts with a [CONTEXT: ...] tag. Use it to decide whether this
section is about Symptoms, Causes, Treatments, Risk Factors, or Comorbidities.
This determines which relations are legal.

STEP 2 — Extract entities.
For each named medical concept produce:
  - id:          sequential "n0", "n1", "n2" ...
  - name:        specific term, UPPERCASE (e.g. LITHIUM, not MEDICATION)
  - type:        exactly one of [CONDITION, SYMPTOM, TREATMENT, MEDICATION,
                 SAFETY_RESOURCE, DEMOGRAPHIC]
  - description: 1 sentence in the context of mental health

TYPE ASSIGNMENT RULES (follow strictly):
  CONDITION   — a diagnosable disorder or disease (Depression, PTSD, Anxiety)
  SYMPTOM     — an experiential sign the patient notices or exhibits
  TREATMENT   — a non-drug intervention (Psychotherapy, CBT, Light Therapy)
  MEDICATION  — a drug or supplement taken to treat a condition
  DEMOGRAPHIC — a population group or statistical risk characteristic
                (Women, Older Adults, Low Birth Weight — NOT a symptom)
  SAFETY_RESOURCE — a helpline, hotline, or crisis service

STEP 3 — Extract relations.
Use ONLY these relations, with the meanings defined below:

  HAS_SYMPTOM          — subject (CONDITION) directly causes or includes
                         target (SYMPTOM) as an experiential sign.
                         target MUST be a SYMPTOM. Never use for medications,
                         demographics, treatments, or comorbid conditions.

  TREATED_BY           — subject (CONDITION) is treated or managed by
                         target (TREATMENT or MEDICATION).

  SUITABLE_FOR         — subject (CONDITION, MEDICATION, or TREATMENT) is
                         specifically relevant to target (DEMOGRAPHIC).
                         Use this when text says a condition "is more common in"
                         or "particularly affects" a group.

  CONTRAINDICATED_FOR  — subject should be avoided for target (DEMOGRAPHIC).

  URGENT_ACTION        — subject (CONDITION or SYMPTOM) requires immediate
                         contact with target (SAFETY_RESOURCE).

  PRESCRIBES           — subject (CONDITION) is prescribed target (MEDICATION).
                         Prefer TREATED_BY unless the text specifically
                         discusses prescription.

WHAT TO DO INSTEAD OF HAS_SYMPTOM for edge cases:
  - "X often co-occurs with Y" (both CONDITIONs) → skip it; no valid relation
  - "X is a risk factor for Y" → skip it; no RISK_FACTOR relation exists
  - "X involves low serotonin" (biological mechanism) → skip it
  - "X is more common in women" → CONDITION -[SUITABLE_FOR]-> DEMOGRAPHIC
  - "X can cause Y" where Y is another CONDITION → skip it
  - "X is a subtype of Y" → skip it

RULES:
  - Entity names must be specific (FATIGUE not SYMPTOM, LITHIUM not MEDICATION)
  - The HAS_SYMPTOM target must always be a SYMPTOM — never MEDICATION,
    TREATMENT, CONDITION, or DEMOGRAPHIC
  - TREATMENT and MEDICATION are NEVER the source of HAS_SYMPTOM
  - If nothing meaningful to extract, return {"entities": [], "relations": []}

OUTPUT: valid JSON only — no markdown fences, no commentary.

{
  "entities": [
    {"id": "n0", "name": "DEPRESSION", "type": "CONDITION", "description": "..."},
    {"id": "n1", "name": "FATIGUE",    "type": "SYMPTOM",   "description": "..."}
  ],
  "relations": [
    {"source": "n0", "target": "n1", "relation": "HAS_SYMPTOM",
     "description": "Depression commonly causes persistent fatigue.", "strength": 9}
  ]
}
"""

# ============================================================
# STORAGE (with resume support)
# ============================================================
custom_entities     = {}   # canonical name → EntityNode
entity_id_to_node   = {}   # UUID → EntityNode  (needed by community detection)
entity_descriptions = {}   # canonical name → best description seen so far
entity_sources      = {}   # canonical name → set of chunk_ids it appeared in
custom_relations    = []   # list of Relation objects (for PropertyGraphIndex)
relation_metadata   = {}   # (subj, rel, obj) → {description, strength, sources}
seen_relations      = set()

# Load pre-processed data if resuming
if pre_loaded_triplets:
    print(f"Loading {len(pre_loaded_triplets)} pre-loaded triplets from checkpoint...")

# NEW: bidirectional index for multipath traversal
# Instead of scanning all relations every time, this gives O(1) lookup
# in both directions for any entity.
#
# Structure:
#   relation_index["DEPRESSION"]["out"] = [("HAS_SYMPTOM", "FATIGUE"), ("TREATED_BY", "CBT")]
#   relation_index["FATIGUE"]["in"]     = [("HAS_SYMPTOM", "DEPRESSION")]
#
# "out" = this entity is the SOURCE of the relation (forward traversal)
# "in"  = this entity is the TARGET of the relation (backward traversal)
relation_index = {}

# NEW: maps chunk UUID → human-readable citation string
# Built here because we have access to node metadata right now.
# Used later by hybrid_query to generate source citations in answers.
chunk_citation_map = {}
for node in enriched_nodes:
    filename = node.metadata.get("file_name", "unknown.md")
    header   = node.metadata.get("header_path", "/")
    chunk_citation_map[node.node_id] = f"{filename} — {header}"

print(f"Built citation map for {len(chunk_citation_map)} chunks")

# Counters
skipped_chunks    = 0
skipped_triplets  = 0
type_conflicts    = 0
garbage_rejected  = 0
semantic_rejected = 0
json_parse_errors = 0

# ============================================================
# PROCESS CHUNKS (RESUMABLE)
# ============================================================
for i, node in enumerate(enriched_nodes[GRAPH_START_INDEX:], start=GRAPH_START_INDEX):
    chunk_id = node.node_id

    print(f"Processing chunk {i+1}/{len(enriched_nodes)} [chunk_id: {chunk_id[:8]}...]")

    try:
        system_msg = ChatMessage(role="system", content=extraction_prompt)
        user_msg   = ChatMessage(role="user",
                                 content=f"Extract from this text:\n\n{node.text}")
        response   = llm.chat([system_msg, user_msg]).message.content.strip()

        response = re.sub(r"^```(?:json)?\s*", "", response)
        response = re.sub(r"\s*```$", "", response)

        if not response:
            skipped_chunks += 1
            # Save progress even on empty response
            with open(GRAPH_PROGRESS_FILE, "a", encoding="utf-8") as pf:
                pf.write(json.dumps({"node_idx": i, "triplets": []}) + "\n")
                pf.flush()
            continue

        try:
            extracted = json.loads(response)
        except json.JSONDecodeError as e:
            json_parse_errors += 1
            print(f"  ⚠️ JSON parse error in chunk {i+1}: {e}")
            # Save progress even on parse error
            with open(GRAPH_PROGRESS_FILE, "a", encoding="utf-8") as pf:
                pf.write(json.dumps({"node_idx": i, "triplets": []}) + "\n")
                pf.flush()
            continue

        raw_entities  = extracted.get("entities", [])
        raw_relations = extracted.get("relations", [])

        # local_id_map is reset every chunk — it's only a bridge between
        # the LLM's temporary n0/n1 IDs and your canonical entity names.
        # It's populated during entity processing and consumed during
        # relation processing, both within this same chunk.
        local_id_map = {}

        # --------------------------------------------------------
        # ENTITY PROCESSING
        # --------------------------------------------------------
        for ent in raw_entities:
            raw_name = ent.get("name", "").strip()
            raw_type = ent.get("type", "").strip().upper()
            local_id = ent.get("id", "").strip()
            desc     = ent.get("description", "").strip()

            if not raw_name or not raw_type or not local_id:
                continue

            canon = canonicalize(raw_name)

            # Always register in local_id_map BEFORE validation.
            # Relations reference these IDs, and we need to resolve
            # them even for entities that don't pass validation —
            # so we can correctly skip the relation too.
            local_id_map[local_id] = canon

            if raw_type not in ALLOWED_TYPES:
                skipped_triplets += 1
                continue

            if is_garbage_entity(canon):
                garbage_rejected += 1
                continue

            forbidden = ALLOWED_TYPES | ALLOWED_RELATIONS | {"SUBJECT", "OBJECT", "ENTITY"}
            if canon in forbidden:
                skipped_triplets += 1
                continue

            if canon not in custom_entities:
                new_node = EntityNode(
                    name=canon,
                    label=raw_type,
                    properties={"description": desc, "sources": [chunk_id]}
                )
                custom_entities[canon]          = new_node
                entity_id_to_node[new_node.id]  = new_node
                entity_descriptions[canon]      = desc
                entity_sources[canon]           = {chunk_id}

                # Initialize bidirectional index entry for this entity
                relation_index[canon] = {"out": [], "in": []}

            else:
                existing = custom_entities[canon]

                resolved = resolve_type(existing.label, raw_type)
                if resolved != existing.label:
                    type_conflicts += 1
                    existing.label = resolved

                if len(desc) > len(entity_descriptions.get(canon, "")):
                    entity_descriptions[canon] = desc
                    existing.properties["description"] = desc

                entity_sources.setdefault(canon, set()).add(chunk_id)
                existing.properties["sources"] = list(entity_sources[canon])

        # --------------------------------------------------------
        # RELATION PROCESSING
        # --------------------------------------------------------
        node_triplets = []
        for rel_data in raw_relations:
            src_local    = rel_data.get("source", "")
            tgt_local    = rel_data.get("target", "")
            rel_label    = rel_data.get("relation", "").strip().upper()
            rel_desc     = rel_data.get("description", "").strip()
            rel_strength = int(rel_data.get("strength", 5))

            subj = local_id_map.get(src_local)
            obj  = local_id_map.get(tgt_local)

            if not subj or not obj:
                skipped_triplets += 1
                continue

            if subj not in custom_entities or obj not in custom_entities:
                skipped_triplets += 1
                continue

            if rel_label not in ALLOWED_RELATIONS:
                skipped_triplets += 1
                continue

            subj_type = custom_entities[subj].label
            obj_type = custom_entities[obj].label
            if not is_valid_triplet(subj_type, rel_label, obj_type):
                semantic_rejected += 1
                continue

            rel_key = (subj, rel_label, obj)

            if rel_key in seen_relations:
                # Relation already exists — just strengthen it
                if rel_key in relation_metadata:
                    if rel_strength > relation_metadata[rel_key]["strength"]:
                        relation_metadata[rel_key]["strength"] = rel_strength
                    relation_metadata[rel_key]["sources"].add(chunk_id)
                continue

            seen_relations.add(rel_key)

            relation = Relation(
                source_id=custom_entities[subj].id,
                target_id=custom_entities[obj].id,
                label=rel_label,
            )
            custom_relations.append(relation)

            relation_metadata[rel_key] = {
                "description": rel_desc,
                "strength":    rel_strength,
                "sources":     {chunk_id},
            }

            # ── UPDATE BIDIRECTIONAL INDEX ──────────────────────
            # Forward edge: subj → obj
            # This lets you ask "what does DEPRESSION connect to?"
            if subj not in relation_index:
                relation_index[subj] = {"out": [], "in": []}
            relation_index[subj]["out"].append((rel_label, obj))

            # Backward edge: obj ← subj
            # This lets you ask "what connects TO FATIGUE?" without
            # scanning every relation in the graph.
            if obj not in relation_index:
                relation_index[obj] = {"out": [], "in": []}
            relation_index[obj]["in"].append((rel_label, subj))
            node_triplets.append({"subj": subj, "subj_type": subj_type, "rel": rel_label, "obj": obj, "obj_type": obj_type})

    except Exception as e:
        print(f"  ⚠️ Chunk {i+1} failed: {e}")
        skipped_chunks += 1
        # Save progress even on exception
        with open(GRAPH_PROGRESS_FILE, "a", encoding="utf-8") as pf:
            pf.write(json.dumps({"node_idx": i, "triplets": []}) + "\n")
            pf.flush()
        continue

    # ── SAVE PROGRESS AFTER EACH CHUNK ────────────────────────
    # Save the triplets extracted from this chunk immediately
    with open(GRAPH_PROGRESS_FILE, "a", encoding="utf-8") as pf:
        pf.write(json.dumps({"node_idx": i, "triplets": node_triplets}) + "\n")
        pf.flush()

print(f"\n✅ Extraction complete!")
print(f"   Unique entities      : {len(custom_entities)}")
print(f"   Unique relations     : {len(custom_relations)}")
print(f"   Bidirectional index  : {len(relation_index)} entities indexed")
print(f"   Chunk citations      : {len(chunk_citation_map)}")
print(f"   Skipped chunks       : {skipped_chunks}")
print(f"   JSON parse errors    : {json_parse_errors}")
print(f"   Skipped (schema)     : {skipped_triplets}")
print(f"   Skipped (garbage)    : {garbage_rejected}")
print(f"   Skipped (semantic)   : {semantic_rejected}")
print(f"   Type conflicts fixed : {type_conflicts}")
print(f"\n💾 Progress saved to {GRAPH_PROGRESS_FILE}")

# Quick sanity check on the bidirectional index:
# Pick one entity and show both directions to verify it's working
sample = "DEPRESSION"
if sample in relation_index:
    out_count = len(relation_index[sample]["out"])
    in_count  = len(relation_index[sample]["in"])
    print(f"\n🔍 Bidirectional index sample — {sample}:")
    print(f"   Outgoing (forward) : {out_count} connections")
    print(f"   Incoming (backward): {in_count} connections")
    print(f"   Example outgoing   : {relation_index[sample]['out'][:3]}")
    print(f"   Example incoming   : {relation_index[sample]['in'][:3]}")

# ============================================================
# BUILD PROPERTY GRAPH INDEX
# ============================================================
print("\nIndexing into Knowledge Graph...")
index = PropertyGraphIndex(nodes=[])
for entity in custom_entities.values():
    index.property_graph_store.upsert_nodes([entity])
index.property_graph_store.upsert_relations(custom_relations)

all_triplets = index.property_graph_store.get_triplets()
print(f"\n✅ Total triplets indexed: {len(all_triplets)}")
for t in all_triplets[:10]:
    print(f"  ({t[0].name} [{t[0].label}]) -[{t[1].label}]-> ({t[2].name} [{t[2].label}])")

# ============================================================
# WRITE TO NEO4J (after extraction or checkpoint load)
# ============================================================
print("\nWriting graph to Neo4j...")
try:
    # Create indexes first (idempotent)
    create_neo4j_indexes(driver)
    
    # Write the graph
    write_graph_to_neo4j(
        driver, 
        custom_entities, 
        seen_relations if 'seen_relations' in dir() else set(relation_metadata.keys()),
        relation_metadata, 
        entity_sources,
        chunk_citation_map if 'chunk_citation_map' in dir() else {}
    )
    print("✅ Graph successfully written to Neo4j!")
except Exception as e:
    print(f"⚠️ Neo4j write failed: {e}")
    print("   (Graph is still available in local variables)")


--- GRAPH BUILDER V4 (with bidirectional index) ---
Built citation map for 165 chunks
Processing chunk 1/165 [chunk_id: 2e270990...]
Processing chunk 2/165 [chunk_id: db6fea32...]
Processing chunk 3/165 [chunk_id: 13ad4dd9...]
Processing chunk 4/165 [chunk_id: 26f8f1ba...]
Processing chunk 5/165 [chunk_id: 1bc16a44...]
Processing chunk 6/165 [chunk_id: e26a2ab0...]
Processing chunk 7/165 [chunk_id: 119507aa...]
Processing chunk 8/165 [chunk_id: df05df71...]
Processing chunk 9/165 [chunk_id: 7205bb9b...]
Processing chunk 10/165 [chunk_id: d77579c1...]
Processing chunk 11/165 [chunk_id: 40649d26...]
Processing chunk 12/165 [chunk_id: 1481f317...]
Processing chunk 13/165 [chunk_id: c35ae0f9...]
Processing chunk 14/165 [chunk_id: 2ddb1699...]
Processing chunk 15/165 [chunk_id: 083d9f5b...]
Processing chunk 16/165 [chunk_id: 410f176a...]
Processing chunk 17/165 [chunk_id: 78833640...]
Processing chunk 18/165 [chunk_id: f2983524...]
Processing chunk 19/165 [chunk_id: b9d06bbb...]
Processing 

In [40]:
print("\n--- CLINICAL FACTS IN THE GRAPH ---")
skipped = 0
for i, rel in enumerate(custom_relations):
    try:
        subj = entity_id_to_node[rel.source_id]
        obj  = entity_id_to_node[rel.target_id]
        print(f"Fact {i+1}: ({subj.name} [{subj.label}]) -[{rel.label}]-> ({obj.name} [{obj.label}])")
    except KeyError as e:
        skipped += 1
        print(f"  ⚠️ Skipped relation {i+1} — orphaned ID: {e}")

if skipped:
    print(f"\n⚠️ {skipped} orphaned relations")


--- CLINICAL FACTS IN THE GRAPH ---
Fact 1: (ANXIETY [CONDITION]) -[HAS_SYMPTOM]-> (STRESS [CONDITION])
Fact 2: (STRESS [CONDITION]) -[HAS_SYMPTOM]-> (EXCESSIVE WORRY [SYMPTOM])
Fact 3: (STRESS [CONDITION]) -[HAS_SYMPTOM]-> (UNEASINESS [SYMPTOM])
Fact 4: (STRESS [CONDITION]) -[HAS_SYMPTOM]-> (TENSION [SYMPTOM])
Fact 5: (STRESS [CONDITION]) -[HAS_SYMPTOM]-> (HEADACHES OR BODY PAIN [SYMPTOM])
Fact 6: (STRESS [CONDITION]) -[HAS_SYMPTOM]-> (HIGH BLOOD PRESSURE [SYMPTOM])
Fact 7: (STRESS [CONDITION]) -[HAS_SYMPTOM]-> (LOSS OF SLEEP [SYMPTOM])
Fact 8: (ANXIETY [CONDITION]) -[HAS_SYMPTOM]-> (EXCESSIVE WORRY [SYMPTOM])
Fact 9: (ANXIETY [CONDITION]) -[HAS_SYMPTOM]-> (UNEASINESS [SYMPTOM])
Fact 10: (ANXIETY [CONDITION]) -[HAS_SYMPTOM]-> (TENSION [SYMPTOM])
Fact 11: (ANXIETY [CONDITION]) -[HAS_SYMPTOM]-> (HEADACHES OR BODY PAIN [SYMPTOM])
Fact 12: (ANXIETY [CONDITION]) -[HAS_SYMPTOM]-> (HIGH BLOOD PRESSURE [SYMPTOM])
Fact 13: (ANXIETY [CONDITION]) -[HAS_SYMPTOM]-> (LOSS OF SLEEP [SYMPTOM])
Fact 

In [25]:
# ============================================================
# CELL 15 — SAVE EXTRACTION CHECKPOINT
# Run this immediately after Cell 14 finishes.
# Saves every data structure Cell 14 produced so you never
# have to re-run the expensive LLM extraction loop again.
# ============================================================

import pickle
import os

CHECKPOINT_DIR = "./checkpoints/extraction_v4"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# What we're saving and why each one matters:
#
# custom_entities     — the graph nodes (name → EntityNode)
# entity_id_to_node   — reverse lookup (UUID → EntityNode), needed by community detection
# entity_descriptions — best description per entity, used for richer query context
# entity_sources      — which chunks each entity came from, powers citations
# custom_relations    — the graph edges (list of Relation objects)
# relation_metadata   — description + strength + sources per relation, used by fidelity check
# relation_index      — bidirectional lookup, powers forward AND backward traversal
# chunk_citation_map  — chunk UUID → filename/header string, powers readable citations
# index               — the PropertyGraphIndex, used by local_query_engine

to_save = {
    "custom_entities":     custom_entities,
    "entity_id_to_node":   entity_id_to_node,
    "entity_descriptions": entity_descriptions,
    "entity_sources":      entity_sources,
    "custom_relations":    custom_relations,
    "relation_metadata":   relation_metadata,
    "relation_index":      relation_index,
    "chunk_citation_map":  chunk_citation_map,
    "index":               index,
}

for filename, obj in to_save.items():
    path = f"{CHECKPOINT_DIR}/{filename}.pkl"
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"✅ Saved {filename}.pkl ({os.path.getsize(path) / 1024:.1f} KB)")

print(f"\n💾 All checkpoints saved to {CHECKPOINT_DIR}/")
print(f"   Run the load cell next time instead of re-running extraction.")

✅ Saved custom_entities.pkl (78.8 KB)
✅ Saved entity_id_to_node.pkl (78.8 KB)
✅ Saved entity_descriptions.pkl (50.0 KB)
✅ Saved entity_sources.pkl (13.8 KB)
✅ Saved custom_relations.pkl (25.2 KB)
✅ Saved relation_metadata.pkl (55.9 KB)
✅ Saved relation_index.pkl (23.5 KB)
✅ Saved chunk_citation_map.pkl (13.6 KB)
✅ Saved index.pkl (146.0 KB)

💾 All checkpoints saved to ./checkpoints/extraction_v4/
   Run the load cell next time instead of re-running extraction.


In [26]:
# ============================================================
# CELL 16 — LOAD EXTRACTION CHECKPOINT
# Run this instead of Cell 14 on any session where you
# don't need to re-extract. Restores everything Cell 14
# produced in about 2 seconds instead of several minutes.
# ============================================================

import pickle
import os

CHECKPOINT_DIR = "./checkpoints/extraction_v4"

expected_files = [
    "custom_entities",
    "entity_id_to_node",
    "entity_descriptions",
    "entity_sources",
    "custom_relations",
    "relation_metadata",
    "relation_index",
    "chunk_citation_map",
    "index",
]

# Verify all files exist before attempting to load anything.
# If even one is missing the graph will be in a broken partial state,
# so we fail loudly here rather than getting a confusing error later.
missing = [
    f for f in expected_files
    if not os.path.exists(f"{CHECKPOINT_DIR}/{f}.pkl")
]
if missing:
    raise FileNotFoundError(
        f"Missing checkpoint files: {missing}\n"
        f"Run Cell 14 and Cell 15 first to generate them."
    )

# Load everything back into the global namespace
for filename in expected_files:
    path = f"{CHECKPOINT_DIR}/{filename}.pkl"
    with open(path, "rb") as f:
        globals()[filename] = pickle.load(f)
    size = os.path.getsize(path) / 1024
    print(f"✅ Loaded {filename}.pkl ({size:.1f} KB)")

# Rebuild local_query_engine from the loaded index.
# This object is not pickled because it contains unpickleable
# internal LlamaIndex state. It takes under a second to recreate.
local_query_engine = index.as_query_engine()

print(f"""
📊 Restoration summary:
   Entities          : {len(custom_entities)}
   Relations         : {len(custom_relations)}
   Bidirectional idx : {len(relation_index)} entities
   Chunk citations   : {len(chunk_citation_map)}
   Entity sources    : {len(entity_sources)}
   Relation metadata : {len(relation_metadata)}
   local_query_engine: ✅ ready

Everything restored. You can now run community detection and hybrid queries.
""")

# Verify the bidirectional index actually works in both directions
sample = "DEPRESSION"
if sample in relation_index:
    print(f"🔍 Bidirectional index check — {sample}:")
    print(f"   Forward  (out): {len(relation_index[sample]['out'])} connections → {relation_index[sample]['out'][:2]}")
    print(f"   Backward (in) : {len(relation_index[sample]['in'])} connections → {relation_index[sample]['in'][:2]}")
else:
    print(f"⚠️  '{sample}' not found in relation_index — check your extraction ran correctly.")

✅ Loaded custom_entities.pkl (78.8 KB)
✅ Loaded entity_id_to_node.pkl (78.8 KB)
✅ Loaded entity_descriptions.pkl (50.0 KB)
✅ Loaded entity_sources.pkl (13.8 KB)
✅ Loaded custom_relations.pkl (25.2 KB)
✅ Loaded relation_metadata.pkl (55.9 KB)
✅ Loaded relation_index.pkl (23.5 KB)
✅ Loaded chunk_citation_map.pkl (13.6 KB)
✅ Loaded index.pkl (146.0 KB)

📊 Restoration summary:
   Entities          : 323
   Relations         : 333
   Bidirectional idx : 323 entities
   Chunk citations   : 165
   Entity sources    : 323
   Relation metadata : 333
   local_query_engine: ✅ ready

Everything restored. You can now run community detection and hybrid queries.

🔍 Bidirectional index check — DEPRESSION:
   Forward  (out): 71 connections → [('HAS_SYMPTOM', 'FATIGUE'), ('HAS_SYMPTOM', 'SADNESS')]
   Backward (in) : 7 connections → [('HAS_SYMPTOM', 'SEASONAL AFFECTIVE DISORDER'), ('HAS_SYMPTOM', 'TRAUMA')]


In [7]:
local_query_engine = index.as_query_engine()
print("✅ local_query_engine ready")

✅ local_query_engine ready


# Community Detection

In [8]:
import importlib, sys
from collections import defaultdict
import numpy as np
import networkx as nx

try:
    sys.modules.setdefault("numpy.char", importlib.import_module("numpy.core.defchararray"))
except Exception:
    sys.modules.setdefault("numpy.char", np.char)

from graspologic.partition import leiden

# ============================================================
# COMMUNITY DETECTION — Leiden Algorithm
# FIX: Use entity UUID as the graph node ID everywhere.
# Previously used entity NAME as node ID but UUID as edge endpoints
# — these never matched, producing a broken disconnected graph.
# ============================================================

G = nx.Graph()

# Add nodes keyed by UUID (matches rel.source_id / rel.target_id)
for entity in custom_entities.values():
    G.add_node(entity.id, name=entity.name, label=entity.label)  # ← FIX: entity.id not eid

for rel in custom_relations:
    if rel.source_id in G and rel.target_id in G:  # safety check
        G.add_edge(rel.source_id, rel.target_id, relation=rel.label)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Isolated node check — should be 0 or very few after the fix
isolated = list(nx.isolates(G))
print(f"Isolated nodes (no edges): {len(isolated)}")

# Run Leiden
partition = leiden(G, random_seed=42)

communities = defaultdict(list)
for node_id, community_id in partition.items():
    communities[community_id].append(node_id)  # node_id is now UUID ✓

print(f"\nLeiden found {len(communities)} communities:\n")

for comm_id, member_ids in sorted(communities.items(), key=lambda x: -len(x[1])):
    # FIX: look up by UUID using entity_id_to_node
    member_names  = [entity_id_to_node[mid].name  for mid in member_ids if mid in entity_id_to_node]
    member_labels = [entity_id_to_node[mid].label for mid in member_ids if mid in entity_id_to_node]

    print(f"Community {comm_id} ({len(member_names)} entities):")
    for name, label in zip(member_names, member_labels):
        print(f"  - {name} [{label}]")

    community_set = set(member_ids)
    internal_rels = [r for r in custom_relations
                     if r.source_id in community_set and r.target_id in community_set]
    print(f"  Internal relations: {len(internal_rels)}")
    print()

Graph built: 309 nodes, 394 edges
Isolated nodes (no edges): 21

Leiden found 13 communities:

Community 5 (49 entities):
  - SUICIDE [SYMPTOM]
  - ACUPUNCTURE [TREATMENT]
  - 911 [SAFETY_RESOURCE]
  - MANIC EPISODE [SYMPTOM]
  - BIPOLAR II DISORDER [CONDITION]
  - DISRUPTIVE MOOD DYSREGULATION DISORDER [CONDITION]
  - HYPOMANIC EPISODE [SYMPTOM]
  - FAMILY FOCUSED THERAPY [TREATMENT]
  - EXTREME RISK TAKING [SYMPTOM]
  - HALLUCINATION [SYMPTOM]
  - BRAIN STRUCTURE [SYMPTOM]
  - SUBSTANCE USE DISORDER [CONDITION]
  - TRANSCRANIAL MAGNETIC STIMULATION [TREATMENT]
  - VALPROATE [MEDICATION]
  - MASSAGE [TREATMENT]
  - BRAIN FUNCTION [SYMPTOM]
  - GENETIC FACTOR [CONDITION]
  - LITHIUM [MEDICATION]
  - NATIONAL INSTITUTE OF MENTAL HEALTH [SAFETY_RESOURCE]
  - HERB [TREATMENT]
  - DELUSION [SYMPTOM]
  - INTERPERSONAL AND SOCIAL RHYTHM THERAPY [TREATMENT]
  - DIFFICULTY MAINTAINING RESPONSIBILITY [SYMPTOM]
  - GENETIC RISK [DEMOGRAPHIC]
  - MOOD EPISODE [SYMPTOM]
  - MAJOR DEPRESSIVE DISORD

C:\Users\derrm\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\graspologic\partition\leiden.py:338: UserWarning: Leiden partitions do not contain all nodes from the input graph because input graph contained isolate nodes.
  warnings.warn(


# Community Summarization

In [ ]:
from llama_index.core.llms import ChatMessage
import json

FORBIDDEN_NAMES = {
    "SYMTOM", "SYMPOM", "DEMOCRAPHIC", "SUBJECT", "OBJECT",
    "CONDITION", "SYMPTOM", "TREATMENT", "DEMOGRAPHIC", "ENTITY",
    "BEHAVIOR", "DEVELOPMENT", "SUPPORT", "SERVICES", "TREATMENTS",
    "SYMPTOMS", "MEDICATION", "SAFETY_RESOURCE"
}

community_summaries = {}
print(f"Summarizing {len(communities)} communities...\n")

for comm_id, member_ids in sorted(communities.items(), key=lambda x: -len(x[1])):

    # FIX: member_ids are UUIDs — use entity_id_to_node, not custom_entities
    members = []
    for mid in member_ids:
        if mid in entity_id_to_node:                          # ← FIX
            e = entity_id_to_node[mid]                        # ← FIX
            if e.name not in FORBIDDEN_NAMES and len(e.name) > 3:
                members.append(f"{e.name} [{e.label}]")

    if len(members) < 2:
        print(f"Community {comm_id} — skipped (too few valid entities)\n")
        continue

    # FIX: valid_ids check uses entity_id_to_node
    valid_ids = {
        mid for mid in member_ids
        if mid in entity_id_to_node                           # ← FIX
        and entity_id_to_node[mid].name not in FORBIDDEN_NAMES  # ← FIX
        and len(entity_id_to_node[mid].name) > 3             # ← FIX
    }

    community_set = set(member_ids)
    internal_rels = []
    for r in custom_relations:
        if r.source_id in community_set and r.target_id in community_set:
            if r.source_id in valid_ids and r.target_id in valid_ids:
                src_name = entity_id_to_node[r.source_id].name  # ← FIX
                tgt_name = entity_id_to_node[r.target_id].name  # ← FIX
                internal_rels.append(f"({src_name}) -[{r.label}]-> ({tgt_name})")

    entities_str  = "\n".join(f"  - {m}" for m in members)
    relations_str = "\n".join(f"  - {r}" for r in internal_rels) if internal_rels else "  (no internal relations)"

    system_msg = ChatMessage(
        role="system",
        content=(
            "You are a medical knowledge graph analyst. Given a cluster of entities and their "
            "relationships from mental health brochures, write a concise summary (3-5 sentences) "
            "that describes:\n"
            "1. The main disorder(s) or topic this community is about\n"
            "2. Key symptoms, treatments, or medications mentioned\n"
            "3. How the entities are related to each other\n"
            "Be factual and clinical. Do not add information not present in the data."
        )
    )
    user_msg = ChatMessage(
        role="user",
        content=(
            f"Summarize this community of medical entities:\n\n"
            f"ENTITIES:\n{entities_str}\n\n"
            f"RELATIONSHIPS:\n{relations_str}"
        )
    )

    response = llm.chat([system_msg, user_msg]).message.content.strip()

    community_summaries[comm_id] = {
        "summary":   response,
        "entities":  members,
        "relations": internal_rels,
        "size":      len(members)
    }

    print(f"Community {comm_id} ({len(members)} entities):")
    print(f"  Summary: {response[:150]}...")
    print()

print(f"\n✅ Generated summaries for {len(community_summaries)} communities.")


Summarizing 24 communities...

Community 16 (78 entities):
  Summary: This community primarily focuses on depression, which is characterized by a range of symptoms including low mood, feelings of worthlessness, memory pr...

Community 18 (67 entities):
  Summary: This community primarily focuses on Bipolar Disorder, characterized by extreme mood changes, including manic states, depressive episodes, and mixed ep...

Community 7 (52 entities):
  Summary: This community focuses on Post-Traumatic Stress Disorder (PTSD) and trauma-related disorders, primarily affecting individuals who have experienced tra...

Community 13 (45 entities):
  Summary: This community primarily focuses on Autism Spectrum Disorder (ASD), highlighting various symptoms associated with the condition, such as difficulty wi...

Community 9 (45 entities):
  Summary: This community focuses on various mood disorders, primarily Bipolar I Disorder, Bipolar II Disorder, and Perimenopausal Depression. Key symptoms assoc...

C

In [ ]:
# Save/Load helpers for community summaries
from pathlib import Path
import json

SUMMARIES_PATH = Path("community_summaries.json")

def save_community_summaries(summaries, path=SUMMARIES_PATH):
    """Persist summaries to disk as JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        str(comm_id): data for comm_id, data in summaries.items()
    }
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)

    print(f"✅ Saved {len(payload)} community summaries to {path}")

save_community_summaries(community_summaries)

In [28]:
from pathlib import Path
import json

SUMMARIES_PATH = Path("community_summaries.json")

def load_community_summaries(path=SUMMARIES_PATH):
    """Load summaries from disk and convert keys back to integers when possible."""
    path = Path(path)
    if not path.exists():
        print(f"⚠️ No summaries file found at {path}")
        return {}

    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)

    loaded = {}
    for key, value in raw.items():
        try:
            key_cast = int(key)
        except (TypeError, ValueError):
            key_cast = key
        loaded[key_cast] = value

    print(f"✅ Loaded {len(loaded)} community summaries from {path}")
    return loaded

# Usage:

community_summaries = load_community_summaries()

✅ Loaded 24 community summaries from community_summaries.json


# Search Engines

In [29]:
# Constants and helper functions for hybrid query

STOPWORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", 
    "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 
    'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 
    'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 
    'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 
    'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 
    'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 
    'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 
    'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 
    'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 
    'same', 'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', "don't", 'should', 
    "should've", 'now', 'd', 'll', 'm', 'o', 're', 've', 'y', 'ain', 'aren', "aren't", 'couldn', 
    "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', 
    "haven't", 'isn', "isn't", 'ma', 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 
    'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 
    'wouldn', "wouldn't"
}

QUERY_STOPWORDS = STOPWORDS | {'symptom', 'symptoms', 'condition', 'conditions', 'disorder', 'disorders', 
                                 'what', 'could', 'might', 'may', 'can', 'would', 'should'}

CRISIS_KEYWORDS = ['suicide', 'suicidal', 'kill myself', 'end my life', 'harm myself', 'self-harm']

def classify_query(query_text: str) -> str:
    """Classify query into local, global, or crisis type."""
    query_lower = query_text.lower()
    
    # Crisis detection
    if any(keyword in query_lower for keyword in CRISIS_KEYWORDS):
        return 'crisis'
    
    # Global search indicators
    global_indicators = ['overview', 'explain', 'what is', 'tell me about', 'describe', 
                        'how does', 'why does', 'general']
    if any(indicator in query_lower for indicator in global_indicators):
        return 'global'
    
    # Default to local search
    return 'local'

def global_query(query_text: str, community_summaries: dict, top_k: int = 3) -> str:
    """Simple global search using community summaries."""
    if not community_summaries:
        return "No community summaries available for global search."
    
    # Simple keyword matching against summaries
    query_lower = query_text.lower()
    query_terms = set(query_lower.split()) - QUERY_STOPWORDS
    
    if not query_terms:
        return "Query too general - please be more specific."
    
    # Score each community summary
    scored_summaries = []
    for comm_id, summary in community_summaries.items():
        summary_lower = summary.lower()
        score = sum(1 for term in query_terms if term in summary_lower)
        if score > 0:
            scored_summaries.append((score, comm_id, summary))
    
    # Sort by score descending
    scored_summaries.sort(reverse=True, key=lambda x: x[0])
    
    if not scored_summaries:
        return "No relevant information found in community summaries."
    
    # Return top K summaries
    results = []
    for score, comm_id, summary in scored_summaries[:top_k]:
        results.append(f"\n--- Community {comm_id} (relevance: {score}) ---\n{summary}")
    
    return "\n".join(results)

def print_result(result, query_type: str = "local"):
    """Pretty print search results."""
    if isinstance(result, str):
        print(result)
        return
    
    print(f"\n{'='*60}")
    print(f"Query Type: {query_type.upper()}")
    print(f"{'='*60}")
    
    if isinstance(result, dict):
        for key, value in result.items():
            print(f"\n{key}:")
            if isinstance(value, (list, set)):
                for item in value:
                    print(f"  - {item}")
            else:
                print(f"  {value}")
    elif isinstance(result, (list, tuple)):
        for i, item in enumerate(result, 1):
            print(f"\n{i}. {item}")
    else:
        print(result)
    
    print(f"\n{'='*60}")

print("Constants and helper functions loaded successfully")


Constants and helper functions loaded successfully


In [30]:
# ============================================================
# CELL A — MULTIPATH RETRIEVAL FUNCTIONS
# These answer two specific question types that simple forward
# lookup cannot handle:
#
#   1. find_shared_and_diverging
#      "What is the difference between depression and bipolar?"
#      → Needs to collect ALL neighbors of each condition and
#        compare them. Uses relation_index["DEPRESSION"]["out"]
#        for forward edges AND relation_index["DEPRESSION"]["in"]
#        for backward edges. O(1) per entity instead of O(n).
#
#   2. reverse_symptom_lookup
#      "I have fatigue and sadness, what could this be?"
#      → Needs to travel BACKWARD through HAS_SYMPTOM edges.
#        Given FATIGUE, find every CONDITION that points TO it.
#        Uses relation_index["FATIGUE"]["in"] directly.
#        O(symptoms) instead of O(all relations).
# ============================================================

def find_shared_and_diverging(entity_names: list,
                               custom_entities: dict,
                               relation_index: dict,
                               relation_metadata: dict):
    """
    Compare 2+ entities by collecting all their neighbors in both
    directions, then finding what they share vs what's unique to each.

    Returns:
        entity_neighbors : {name: {"out": {rel: {neighbors}}, "in": {rel: {neighbors}}}}
        shared           : {rel_label: set of names ALL entities share}
        unique           : {name: {rel_label: set of names only IT has}}
    """
    entity_neighbors = {}

    for name in entity_names:
        if name not in custom_entities:
            print(f"  ⚠️ '{name}' not found in graph — skipping")
            continue

        if name not in relation_index:
            # Entity exists but has no edges at all
            entity_neighbors[name] = {"out": {}, "in": {}}
            continue

        out_neighbors = {}  # rel_label → set of neighbor names (forward edges)
        in_neighbors  = {}  # rel_label → set of neighbor names (backward edges)

        # Forward edges: things this entity points TO
        # e.g. DEPRESSION --[HAS_SYMPTOM]--> FATIGUE
        # relation_index["DEPRESSION"]["out"] = [("HAS_SYMPTOM", "FATIGUE"), ...]
        for (rel_label, neighbor_name) in relation_index[name]["out"]:
            out_neighbors.setdefault(rel_label, set()).add(neighbor_name)

        # Backward edges: things that point TO this entity
        # e.g. CBT --[TREATED_BY]--> DEPRESSION means DEPRESSION has an incoming TREATED_BY from CBT
        # relation_index["DEPRESSION"]["in"] = [("TREATED_BY", "CBT"), ...]
        for (rel_label, neighbor_name) in relation_index[name]["in"]:
            # Prefix with "<-" so we can tell forward from backward in the output
            in_neighbors.setdefault(f"<-{rel_label}", set()).add(neighbor_name)

        entity_neighbors[name] = {**out_neighbors, **in_neighbors}

    if len(entity_neighbors) < 2:
        return entity_neighbors, {}, {}

    # Collect every relation type seen across any entity
    all_relation_types = set()
    for neighbors in entity_neighbors.values():
        all_relation_types.update(neighbors.keys())

    shared = {}  # rel_label → names that ALL entities share
    unique = {}  # entity_name → rel_label → names only IT has

    for rel_type in all_relation_types:
        # Get this relation's neighbor set for each entity
        # If an entity has no neighbors of this type, use empty set
        sets = [
            entity_neighbors[name].get(rel_type, set())
            for name in entity_names
            if name in entity_neighbors
        ]

        if not sets:
            continue

        # Shared = intersection: names present in EVERY entity's set
        intersection = sets[0].copy()
        for s in sets[1:]:
            intersection &= s
        if intersection:
            shared[rel_type] = intersection

        # Unique = what each entity has that no other entity has
        valid_names = [n for n in entity_names if n in entity_neighbors]
        for i, name in enumerate(valid_names):
            others_union = set()
            for j, s in enumerate(sets):
                if j != i:
                    others_union |= s
            only_mine = sets[i] - others_union
            if only_mine:
                unique.setdefault(name, {})[rel_type] = only_mine

    return entity_neighbors, shared, unique


def reverse_symptom_lookup(symptom_names: list,
                            custom_entities: dict,
                            relation_index: dict):
    """
    Given symptom names, find every CONDITION that claims those symptoms,
    ranked by how many of the input symptoms they match.

    How it works:
        For each symptom, look at relation_index[symptom]["in"].
        That list contains every entity that has an edge POINTING TO this symptom.
        Filter to only those where the source is a CONDITION and the
        relation is HAS_SYMPTOM. Those are your candidate conditions.

    Returns:
        [(condition_name, [matched_symptom1, matched_symptom2, ...]), ...]
        sorted by number of matched symptoms, descending.
    """
    # Canonicalize all input symptom names so they match what's in the graph
    symptom_set = {canonicalize(s) for s in symptom_names}

    # Filter to only symptoms that actually exist in the graph
    # (avoids silently missing symptoms due to naming differences)
    found_symptoms    = symptom_set & set(relation_index.keys())
    missing_symptoms  = symptom_set - found_symptoms
    if missing_symptoms:
        print(f"  ⚠️ These symptoms weren't found in the graph: {missing_symptoms}")
        print(f"     Found: {found_symptoms}")

    condition_matches = {}  # condition_name → list of matched symptom names

    for symptom_name in found_symptoms:
        if symptom_name not in relation_index:
            continue

        # relation_index[symptom]["in"] gives us all edges that point TO this symptom
        # Each entry is (rel_label, source_entity_name)
        for (rel_label, source_name) in relation_index[symptom_name]["in"]:

            # We only care about HAS_SYMPTOM edges coming from CONDITIONs
            if rel_label != "HAS_SYMPTOM":
                continue
            if source_name not in custom_entities:
                continue
            if custom_entities[source_name].label != "CONDITION":
                continue

            condition_matches.setdefault(source_name, []).append(symptom_name)

    # Sort by number of matched symptoms (most matches first)
    ranked = sorted(condition_matches.items(), key=lambda x: -len(x[1]))
    return ranked


print("✅ Multipath retrieval functions loaded")
print("   find_shared_and_diverging → uses relation_index (O(1) lookup)")
print("   reverse_symptom_lookup    → uses relation_index (O(1) lookup)")

✅ Multipath retrieval functions loaded
   find_shared_and_diverging → uses relation_index (O(1) lookup)
   reverse_symptom_lookup    → uses relation_index (O(1) lookup)


In [33]:
# ============================================================
# CELL B — MISSING CONSTANTS + classify_query FIX
# These are required by the structural tests in Cell C.
# ============================================================

CRISIS_RESPONSE = (
    "It sounds like you may be in crisis. Please reach out for help immediately.\n"
    "🆘 988 Suicide & Crisis Lifeline: call or text 988\n"
    "🆘 Crisis Text Line: text HOME to 741741\n"
    "🆘 Emergency services: 911"
)

OUT_OF_SCOPE_RESPONSE = (
    "I'm a mental health information assistant. I can only answer questions related "
    "to mental health conditions, symptoms, and treatments. Please ask me something "
    "in that area."
)

MENTAL_HEALTH_KEYWORDS = {
    "depression", "anxiety", "ptsd", "trauma", "bipolar", "stress", "autism",
    "asd", "adhd", "mood", "disorder", "mental", "health", "therapy", "treatment",
    "symptom", "panic", "ocd", "schizophrenia", "psychosis", "phobia", "sad",
    "seasonal", "affective", "perinatal", "postpartum", "fatigue", "insomnia",
    "hypersomnia", "suicidal", "self-harm", "hallucination", "delusion"
}

def classify_query(query_text: str) -> str:
    """
    Classify a query into one of three categories:
      - 'CRISIS'       : user mentions self-harm or suicidal intent
      - 'OUT_OF_SCOPE' : query is unrelated to mental health
      - 'IN_SCOPE'     : a normal mental health question
    """
    query_lower = query_text.lower()

    # 1. Crisis detection — highest priority
    if any(keyword in query_lower for keyword in CRISIS_KEYWORDS):
        return "CRISIS"

    # 2. In-scope detection — does it touch any mental health keyword?
    words = set(query_lower.split())
    if words & MENTAL_HEALTH_KEYWORDS:
        return "IN_SCOPE"

    # 3. Otherwise treat as out-of-scope
    return "OUT_OF_SCOPE"

print("✅ CRISIS_RESPONSE, OUT_OF_SCOPE_RESPONSE, MENTAL_HEALTH_KEYWORDS defined")
print("✅ classify_query updated → returns CRISIS | IN_SCOPE | OUT_OF_SCOPE")

✅ CRISIS_RESPONSE, OUT_OF_SCOPE_RESPONSE, MENTAL_HEALTH_KEYWORDS defined
✅ classify_query updated → returns CRISIS | IN_SCOPE | OUT_OF_SCOPE


In [34]:
# ============================================================
# ENHANCED HYBRID QUERY WITH NEO4J INTEGRATION
# Automatically detects question type and uses appropriate retrieval
# ============================================================

CRISIS_RESPONSE = """I'm concerned about what you're sharing. Please reach out to a crisis helpline:
- National Suicide Prevention Lifeline: 988 (US)
- Crisis Text Line: Text HOME to 741741
- International Association for Suicide Prevention: https://www.iasp.info/resources/Crisis_Centres/
You don't have to face this alone. Please talk to someone who can help."""

OUT_OF_SCOPE_RESPONSE = """I'm designed to provide information about mental health conditions, symptoms, and treatments. 
Your question appears to be outside my area of expertise. Please consult appropriate resources or rephrase your question 
to focus on mental health topics."""

def hybrid_query(question, community_summaries=None, local_query_engine=None, llm=None,
                 entity_sources=None, chunk_citation_map=None,
                 custom_entities=None, entity_id_to_node=None,
                 custom_relations=None, relation_metadata=None):
    """
    Enhanced hybrid query with automatic question type detection.
    Now uses Neo4j for graph queries.
    
    Supports three retrieval patterns:
    1. Forward lookup: "What are the symptoms of X?"
    2. Reverse symptom lookup: "I have X and Y, what could this be?"
    3. Comparison: "What's the difference between X and Y?"
    """
    
    # Use globals if not provided
    if community_summaries is None:
        community_summaries = globals().get("community_summaries", {})
    if custom_entities is None:
        custom_entities = globals().get("custom_entities", {})
    if entity_id_to_node is None:
        entity_id_to_node = globals().get("entity_id_to_node", {})
    if custom_relations is None:
        custom_relations = globals().get("custom_relations", [])
    if relation_metadata is None:
        relation_metadata = globals().get("relation_metadata", {})
    if entity_sources is None:
        entity_sources = globals().get("entity_sources", {})
    if chunk_citation_map is None:
        chunk_citation_map = globals().get("chunk_citation_map", {})
    
    print(f"❓ Question: {question}\n")
    
    # --- Step 1: Classify the query ---
    query_class = classify_query(question)
    
    if query_class == "CRISIS":
        return {
            "answer": CRISIS_RESPONSE,
            "local_result": "",
            "local_was_useful": False,
            "global_result": "",
            "multipath_result": "",
            "communities_used": [],
            "citations": [],
            "out_of_scope": False,
            "is_crisis": True,
            "query_type": "crisis"
        }
    
    if query_class == "OUT_OF_SCOPE":
        return {
            "answer": OUT_OF_SCOPE_RESPONSE,
            "local_result": "",
            "local_was_useful": False,
            "global_result": "",
            "multipath_result": "",
            "communities_used": [],
            "citations": [],
            "out_of_scope": True,
            "is_crisis": False,
            "query_type": "out_of_scope"
        }
    
    # --- Step 2: Detect question type and extract entities ---
    question_lower = question.lower()
    
    # Find mentioned conditions
    mentioned_conditions = [
        name for name in custom_entities
        if name.lower() in question_lower
        and custom_entities[name].label == "CONDITION"
    ]
    
    # Find mentioned symptoms
    mentioned_symptoms = [
        name for name in custom_entities
        if name.lower() in question_lower
        and custom_entities[name].label == "SYMPTOM"
    ]
    
    # Detect comparison patterns
    comparison_patterns = ["difference between", "compare", "vs", "versus", "distinguish"]
    is_comparison = any(p in question_lower for p in comparison_patterns)
    
    # Detect symptom-to-condition patterns
    symptom_patterns = ["i have", "i feel", "experiencing", "suffering from", "what could", "what might"]
    is_symptom_query = any(p in question_lower for p in symptom_patterns) and mentioned_symptoms
    
    print(f"   Detected conditions: {mentioned_conditions}")
    print(f"   Detected symptoms: {mentioned_symptoms}")
    
    # --- Step 3: Route to appropriate retrieval ---
    multipath_result = ""
    query_type = "local"
    
    # Case 1: Comparison query (two or more conditions)
    if is_comparison and len(mentioned_conditions) >= 2:
        query_type = "comparison"
        cond_a, cond_b = mentioned_conditions[0], mentioned_conditions[1]
        print(f"   🔀 Using COMPARISON retrieval for {cond_a} vs {cond_b}")
        
        try:
            comparison = neo4j_comparison(driver, cond_a, cond_b)
            
            shared = comparison.get("shared", {})
            unique_a = comparison.get("unique_a", {})
            unique_b = comparison.get("unique_b", {})
            
            multipath_result = f"Comparison: {cond_a} vs {cond_b}\n\n"
            
            if shared:
                multipath_result += "SHARED:\n"
                for rel_type, entities in shared.items():
                    multipath_result += f"  {rel_type}: {', '.join(entities)}\n"
            
            multipath_result += f"\nUNIQUE TO {cond_a}:\n"
            for rel_type, entities in unique_a.items():
                multipath_result += f"  {rel_type}: {', '.join(entities)}\n"
            
            multipath_result += f"\nUNIQUE TO {cond_b}:\n"
            for rel_type, entities in unique_b.items():
                multipath_result += f"  {rel_type}: {', '.join(entities)}\n"
                
        except Exception as e:
            print(f"   ⚠️ Comparison failed: {e}")
            multipath_result = f"Could not compare: {e}"
    
    # Case 2: Reverse symptom lookup (symptoms → conditions)
    elif is_symptom_query and mentioned_symptoms:
        query_type = "reverse_symptom"
        print(f"   🔍 Using REVERSE SYMPTOM lookup for {mentioned_symptoms}")
        
        try:
            ranked = neo4j_reverse_lookup(driver, mentioned_symptoms)
            
            if ranked:
                multipath_result = f"Conditions matching symptoms ({', '.join(mentioned_symptoms)}):\n\n"
                for i, r in enumerate(ranked[:5], 1):
                    multipath_result += f"{i}. {r['condition']} (matches: {r['match_count']})\n"
                    multipath_result += f"   Matched: {', '.join(r['matched_symptoms'])}\n"
                    if r.get('description'):
                        multipath_result += f"   {r['description'][:200]}...\n"
            else:
                multipath_result = "No conditions found matching those symptoms."
                
        except Exception as e:
            print(f"   ⚠️ Reverse lookup failed: {e}")
            multipath_result = f"Could not perform reverse lookup: {e}"
    
    # Case 3: Forward lookup (condition → symptoms/treatments)
    elif mentioned_conditions:
        query_type = "forward_lookup"
        condition = mentioned_conditions[0]
        print(f"   ➡️ Using FORWARD LOOKUP for {condition}")
        
        try:
            relations = neo4j_forward_lookup(driver, condition)
            
            if relations:
                multipath_result = f"Information about {condition}:\n\n"
                for rel_type, targets in relations.items():
                    multipath_result += f"{rel_type}:\n"
                    for t in targets[:10]:
                        strength_str = f" (strength: {t['strength']})" if t.get('strength') else ""
                        multipath_result += f"  - {t['name']}{strength_str}\n"
            else:
                multipath_result = f"No direct relationships found for {condition}."
                
        except Exception as e:
            print(f"   ⚠️ Forward lookup failed: {e}")
            multipath_result = f"Could not perform forward lookup: {e}"
    
    # --- Step 4: Neo4j text search for additional context ---
    local_text = ""
    try:
        keywords = [w for w in question.upper().split() if len(w) > 3 and w not in QUERY_STOPWORDS]
        if keywords:
            local_rows = neo4j_text_query(driver, keywords[:5], limit=10)
            if local_rows:
                local_text = "\n".join(
                    f"({r['name']} [{r['type']}]): {r['description'][:200] if r.get('description') else 'No description'}" 
                    for r in local_rows
                )
    except Exception as e:
        print(f"   ⚠️ Text search failed: {e}")
    
    # --- Step 5: Global search (community summaries) ---
    global_text = ""
    communities_used = []
    
    if community_summaries:
        try:
            global_result = global_query(question, community_summaries, top_k=3)
            if global_result and "No relevant" not in global_result:
                global_text = global_result
                # Extract community IDs from result
                import re
                communities_used = re.findall(r'Community (\d+)', global_result)
        except Exception as e:
            print(f"   ⚠️ Global search failed: {e}")
    
    # --- Step 6: Synthesize answer with LLM ---
    context_parts = []
    
    if multipath_result:
        context_parts.append(f"GRAPH KNOWLEDGE:\n{multipath_result}")
    
    if local_text:
        context_parts.append(f"ENTITY INFORMATION:\n{local_text}")
    
    if global_text:
        context_parts.append(f"COMMUNITY CONTEXT:\n{global_text}")
    
    combined_context = "\n\n".join(context_parts)
    
    if llm and combined_context:
        synthesis_prompt = f"""Based on the following information about mental health, answer the question.
Be helpful, accurate, and compassionate. If the information is insufficient, say so.

CONTEXT:
{combined_context}

QUESTION: {question}

ANSWER:"""
        
        try:
            answer = llm.complete(synthesis_prompt).text.strip()
        except Exception as e:
            answer = f"I found relevant information but couldn't synthesize an answer: {e}\n\nRaw information:\n{combined_context[:1000]}"
    elif combined_context:
        answer = f"Here's what I found:\n\n{combined_context}"
    else:
        answer = "I couldn't find specific information about that. Please try rephrasing your question."
    
    # --- Step 7: Gather citations ---
    citations = []
    # Add citation logic if needed
    
    print(f"\n   Query type: {query_type}")
    print(f"   Answer length: {len(answer)} chars")
    
    return {
        "answer": answer,
        "local_result": local_text,
        "local_was_useful": bool(local_text),
        "global_result": global_text,
        "multipath_result": multipath_result,
        "communities_used": communities_used,
        "citations": citations,
        "out_of_scope": False,
        "is_crisis": False,
        "query_type": query_type
    }


print("✅ Neo4j-integrated hybrid_query loaded")


✅ hybrid_query loaded
   Paths: comparison | differential_diagnosis | forward_lookup | general
   Multipath now uses relation_index — no more full scans


In [35]:
# ============================================================
# CELL C — TESTS + LIVE QUERY
# Run this to verify everything works end to end.
# ============================================================

# ── Quick structural tests (no LLM calls) ──────────────────
print("=" * 60)
print("🧪 STRUCTURAL TESTS")
print("=" * 60)

passed = 0
failed = 0

def check(name, condition):
    global passed, failed
    if condition:
        print(f"  ✅ {name}")
        passed += 1
    else:
        print(f"  ❌ FAILED: {name}")
        failed += 1

# These must all be in memory before hybrid_query will work
check("custom_entities populated",    len(custom_entities) > 0)
check("relation_index populated",     len(relation_index) > 0)
check("entity_sources populated",     len(entity_sources) > 0)
check("chunk_citation_map populated", len(chunk_citation_map) > 0)
check("relation_metadata populated",  len(relation_metadata) > 0)
check("community_summaries loaded",   len(community_summaries) > 0)
check("local_query_engine exists",    "local_query_engine" in globals())
check("CRISIS_RESPONSE defined",      "CRISIS_RESPONSE" in globals())
check("OUT_OF_SCOPE_RESPONSE defined","OUT_OF_SCOPE_RESPONSE" in globals())
check("MENTAL_HEALTH_KEYWORDS defined","MENTAL_HEALTH_KEYWORDS" in globals())

# Test relation_index has both directions for a known entity
if "DEPRESSION" in relation_index:
    check("relation_index has forward edges for DEPRESSION",
          len(relation_index["DEPRESSION"]["out"]) > 0)
    check("relation_index has backward edges for DEPRESSION",
          len(relation_index["DEPRESSION"]["in"]) > 0)
    print(f"\n  DEPRESSION forward  : {relation_index['DEPRESSION']['out'][:2]}")
    print(f"  DEPRESSION backward : {relation_index['DEPRESSION']['in'][:2]}")
else:
    check("DEPRESSION in relation_index", False)

# Test classify_query returns the right values
check("classify_query: crisis",       classify_query("I want to kill myself") == "CRISIS")
check("classify_query: out of scope", classify_query("best pizza recipe")     == "OUT_OF_SCOPE")
check("classify_query: in scope",     classify_query("What is depression?")   == "IN_SCOPE")

# Test reverse_symptom_lookup uses the index correctly
ranked = reverse_symptom_lookup(["FATIGUE"], custom_entities, relation_index)
check("reverse_symptom_lookup returns results", len(ranked) > 0)
if ranked:
    print(f"\n  Top conditions for FATIGUE: {[r[0] for r in ranked[:3]]}")

# Test find_shared_and_diverging works for two known conditions
_, shared, unique = find_shared_and_diverging(
    ["DEPRESSION", "ANXIETY"],
    custom_entities, relation_index, relation_metadata
)
check("find_shared_and_diverging returns shared", len(shared) > 0 or len(unique) > 0)
if shared:
    print(f"\n  Shared between DEPRESSION and ANXIETY:")
    for rel, names in list(shared.items())[:2]:
        print(f"    {rel}: {list(names)[:3]}")

print(f"\n{'=' * 60}")
print(f"  {passed} passed / {failed} failed")
print("=" * 60)

if failed > 0:
    print("\n⚠️  Fix failures above before running live queries.")
    print("   Most likely cause: a previous cell didn't run, or checkpoint is missing.")
else:
    print("\n✅ All checks passed — running live queries...\n")

    # ── Live query 1: Differential diagnosis (backward traversal) ──
    print("\n" + "=" * 60)
    print("LIVE QUERY 1 — Differential diagnosis")
    print("=" * 60)
    result1 = hybrid_query(
        "I have fatigue and sadness, what conditions could this be?",
        community_summaries, local_query_engine, llm
    )
    print_result(result1)

    # ── Live query 2: Comparison (shared + unique features) ────────
    print("\n" + "=" * 60)
    print("LIVE QUERY 2 — Comparison")
    print("=" * 60)
    result2 = hybrid_query(
        "What is the difference between depression and bipolar disorder?",
        community_summaries, local_query_engine, llm
    )
    print_result(result2)

    # ── Live query 3: Forward lookup ───────────────────────────────
    print("\n" + "=" * 60)
    print("LIVE QUERY 3 — Forward lookup")
    print("=" * 60)
    result3 = hybrid_query(
        "What are the symptoms of PTSD?",
        community_summaries, local_query_engine, llm
    )
    print_result(result3)

🧪 STRUCTURAL TESTS
  ✅ custom_entities populated
  ✅ relation_index populated
  ✅ entity_sources populated
  ✅ chunk_citation_map populated
  ✅ relation_metadata populated
  ✅ community_summaries loaded
  ✅ local_query_engine exists
  ✅ CRISIS_RESPONSE defined
  ✅ OUT_OF_SCOPE_RESPONSE defined
  ✅ MENTAL_HEALTH_KEYWORDS defined
  ✅ relation_index has forward edges for DEPRESSION
  ✅ relation_index has backward edges for DEPRESSION

  DEPRESSION forward  : [('HAS_SYMPTOM', 'FATIGUE'), ('HAS_SYMPTOM', 'SADNESS')]
  DEPRESSION backward : [('HAS_SYMPTOM', 'SEASONAL AFFECTIVE DISORDER'), ('HAS_SYMPTOM', 'TRAUMA')]
  ✅ classify_query: crisis
  ✅ classify_query: out of scope
  ❌ FAILED: classify_query: in scope
  ✅ reverse_symptom_lookup returns results

  Top conditions for FATIGUE: ['DEPRESSION', 'PERINATAL DEPRESSION', 'POSTPARTUM DEPRESSION']
  ✅ find_shared_and_diverging returns shared

  Shared between DEPRESSION and ANXIETY:
    TREATED_BY: ['EXERCISE']
    HAS_SYMPTOM: ['STRESS']

  1

In [ ]:
# ============================================================
# TEST QUERY CELL
# Run this after loading checkpoints to verify everything works
# ============================================================

print("=" * 60)
print("GRAPH RAG TEST QUERIES")
print("=" * 60)

# Quick sanity check - verify data is loaded
print(f"\n📊 Data Status:")
print(f"   custom_entities:    {len(custom_entities)} entities")
print(f"   custom_relations:   {len(custom_relations)} relations")
print(f"   entity_id_to_node:  {len(entity_id_to_node)} mappings")

# Show some sample entities
print(f"\n📋 Sample Entities (first 10):")
for i, (name, node) in enumerate(list(custom_entities.items())[:10]):
    print(f"   {i+1}. {name} [{node.label}]")

# Show some sample relations
print(f"\n🔗 Sample Relations (first 10):")
for i, rel in enumerate(custom_relations[:10]):
    src = entity_id_to_node.get(rel.source_id)
    tgt = entity_id_to_node.get(rel.target_id)
    if src and tgt:
        print(f"   {i+1}. {src.name} --[{rel.label}]--> {tgt.name}")

print("\n" + "=" * 60)
print("TEST QUERIES")
print("=" * 60)

# ============================================================
# Test Query 1: Forward lookup (symptoms of a condition)
# ============================================================
print("\n🔍 Test 1: Forward Lookup")
print("-" * 40)
test_q1 = "What are the symptoms of depression?"
print(f"Query: {test_q1}")

try:
    result1 = hybrid_query(
        test_q1,
        community_summaries,
        local_query_engine,
        llm
    )
    print(f"\nQuery Type: {result1.get('query_type', 'unknown')}")
    print(f"Answer: {result1.get('answer', 'No answer')[:500]}...")
except Exception as e:
    print(f"Error: {e}")

# ============================================================
# Test Query 2: Reverse symptom lookup (differential diagnosis)
# ============================================================
print("\n\n🔍 Test 2: Reverse Symptom Lookup")
print("-" * 40)
test_q2 = "I have fatigue and sadness, what could this be?"
print(f"Query: {test_q2}")

try:
    result2 = hybrid_query(
        test_q2,
        community_summaries,
        local_query_engine,
        llm
    )
    print(f"\nQuery Type: {result2.get('query_type', 'unknown')}")
    print(f"Answer: {result2.get('answer', 'No answer')[:500]}...")
except Exception as e:
    print(f"Error: {e}")

# ============================================================
# Test Query 3: Comparison query
# ============================================================
print("\n\n🔍 Test 3: Comparison Query")
print("-" * 40)
test_q3 = "What is the difference between anxiety and depression?"
print(f"Query: {test_q3}")

try:
    result3 = hybrid_query(
        test_q3,
        community_summaries,
        local_query_engine,
        llm
    )
    print(f"\nQuery Type: {result3.get('query_type', 'unknown')}")
    print(f"Answer: {result3.get('answer', 'No answer')[:500]}...")
except Exception as e:
    print(f"Error: {e}")

# ============================================================
# Test Query 4: Treatment query
# ============================================================
print("\n\n🔍 Test 4: Treatment Query")
print("-" * 40)
test_q4 = "How is anxiety treated?"
print(f"Query: {test_q4}")

try:
    result4 = hybrid_query(
        test_q4,
        community_summaries,
        local_query_engine,
        llm
    )
    print(f"\nQuery Type: {result4.get('query_type', 'unknown')}")
    print(f"Answer: {result4.get('answer', 'No answer')[:500]}...")
except Exception as e:
    print(f"Error: {e}")

print("\n" + "=" * 60)
print("TEST COMPLETE")
print("=" * 60)


In [ ]:
# ============================================================
# NEO4J TEST QUERIES
# Run this to verify Neo4j integration is working
# ============================================================

print("=" * 60)
print("NEO4J GRAPH TEST")
print("=" * 60)

# Test 1: Count nodes and relationships
print("\n📊 Graph Statistics:")
with driver.session(database=NEO4J_DATABASE) as session:
    node_count = session.run("MATCH (n:Entity) RETURN count(n) AS count").single()["count"]
    rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS count").single()["count"]
    print(f"   Nodes: {node_count}")
    print(f"   Relationships: {rel_count}")

# Test 2: Show entity types
print("\n📋 Entity Types:")
with driver.session(database=NEO4J_DATABASE) as session:
    types = session.run(
        "MATCH (n:Entity) RETURN n.type AS type, count(*) AS count ORDER BY count DESC"
    ).data()
    for t in types:
        print(f"   {t['type']}: {t['count']}")

# Test 3: Show relationship types
print("\n🔗 Relationship Types:")
with driver.session(database=NEO4J_DATABASE) as session:
    rels = session.run(
        "MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count ORDER BY count DESC"
    ).data()
    for r in rels:
        print(f"   {r['type']}: {r['count']}")

# Test 4: Sample forward lookup
print("\n🔍 Test Forward Lookup (Depression):")
depression_relations = neo4j_forward_lookup(driver, "DEPRESSION")
for rel_type, targets in list(depression_relations.items())[:3]:
    print(f"   {rel_type}:")
    for t in targets[:3]:
        print(f"      - {t['name']} (strength: {t['strength']})")

# Test 5: Sample reverse lookup
print("\n🔍 Test Reverse Lookup (fatigue, sadness):")
conditions = neo4j_reverse_lookup(driver, ["fatigue", "sadness"])
for c in conditions[:5]:
    print(f"   {c['condition']}: matched {c['match_count']} symptoms - {c['matched_symptoms']}")

# Test 6: Sample comparison
print("\n🔍 Test Comparison (Depression vs Anxiety):")
comparison = neo4j_comparison(driver, "DEPRESSION", "ANXIETY")
print(f"   Shared symptoms: {len(comparison.get('shared', {}).get('HAS_SYMPTOM', set()))}")
print(f"   Unique to Depression: {len(comparison.get('unique_a', {}).get('HAS_SYMPTOM', set()))}")
print(f"   Unique to Anxiety: {len(comparison.get('unique_b', {}).get('HAS_SYMPTOM', set()))}")

print("\n" + "=" * 60)
print("NEO4J TEST COMPLETE ✅")
print("=" * 60)

